# PROMETHEUS-EBM: Epistemic Metacognition Benchmark
### Portable Edition — Runs on Kaggle · Google Colab · JupyterLab · Any Python Env

PROMETHEUS-EBM evaluates whether frontier language models can distinguish answerable questions from unanswerable ones — measuring epistemic metacognition, not just accuracy.

## Required Input Files

Ensure the following dataset files are available before execution:

| File | Required By | Contents |
|---|---|---|
| `prometheus_200_multimodel_dataset.json` | STANDARD, EXTENDED | Core 200-item multi-model evaluation set |
| `prometheus_1000_dataset.json` | DEEP_PROBE | 1000-item single-model deep diagnostic set |
| `probe_ambiguity.json` | Epoch-2 (P01) | Ambiguity stress probes (UNDERDETERMINED class) |
| `probe_contradictions.json` | Epoch-2 (P01) | Contradiction stress probes (CONTRADICTORY class) |

## Execution Options

### Option A: Kaggle (Primary — uses kbench for model routing)
1. Attach required input files above.
2. In Cell C04: set `BENCHMARK_MODE` and `EXECUTION_MODE = "kaggle"`.
3. Run all cells top to bottom.

### Option B: Google Colab
1. Upload dataset JSON files to the Colab session or mount Google Drive.
2. In Cell C04: set `EXECUTION_MODE = "api"` and fill `MODEL_API_KEY` + `API_PROVIDER`.
3. Run all cells top to bottom.

### Option C: Local JupyterLab / Any Python Environment
1. Place dataset JSON files in the notebook directory.
2. In Cell C04: set `EXECUTION_MODE = "api"` and configure your provider.
3. Run all cells top to bottom.

### Option D: SDK (Python script, no notebook needed)
```bash
pip install prometheus-ebm
```
```python
from prometheus_ebm import RunConfig, PrometheusRunner, resolve_models_from_indices, KAGGLE_MODEL_CATALOG

# Select models by 1-based index (same as C04 below)
models = resolve_models_from_indices([25, 5, 7, 11, 18])
config = RunConfig(
    mode='standard',
    provider='openrouter',
    api_key='YOUR_KEY',
    models=models,
)
runner = PrometheusRunner(config)
results = runner.run()
```

## Execution Modes
- `kaggle` (default): Kaggle-hosted model inference via kbench.
- `api`: External provider (OpenRouter, OpenAI, Anthropic) — for Colab/local.
- `offline_validation`: Synthetic responses only. Pipeline integrity checks.

## Final Output Targets
- `Final_Output_main.csv` / `.json`
- `prometheus_results_export.zip`

No pre-baked outputs stored; all cells run from a clean state.

## Research Methodology

PROMETHEUS-EBM measures whether models know when they can and cannot answer reliably — not just whether they get answers right.

### Solvability Taxonomy

| Class | Definition |
|---|---|
| `DETERMINATE` | Exactly one correct answer exists given the available information. |
| `UNDERDETERMINED` | Multiple valid interpretations or answers exist; no single answer is uniquely correct. |
| `INSUFFICIENT` | Key information is missing; the question cannot be answered from the given premises. |
| `CONTRADICTORY` | The premises conflict internally; no coherent answer is possible. |

### Core Metrics
- **ECI**: Epistemic Calibration Index (composite; higher is better)
- **HGI**: Hysteresis Gap Index (internal inconsistency; lower is better)
- **Brier decomposition**: Reliability, Resolution, Uncertainty
- **Type-2 D-prime**: Metacognitive discrimination ability

### Interpretation Notes
- Results from `kaggle` or `api` mode are the only valid basis for model-quality claims.
- `offline_validation` mode uses deterministic synthetic responses and is for pipeline integrity checks only.
- High ECI with low D-prime is a diagnostic warning: incidental correctness rather than genuine metacognitive awareness.

In [ ]:
# [C03] Environment Setup — Platform-Adaptive
import sys
import os
import subprocess

# ── Platform Detection ────────────────────────────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = 'google.colab' in sys.modules
IS_LOCAL  = not IS_KAGGLE and not IS_COLAB

PLATFORM  = 'kaggle' if IS_KAGGLE else ('colab' if IS_COLAB else 'local')
print(f'Platform detected: {PLATFORM}')

# ── Dependency Installation ───────────────────────────────────────────────────
# Install prometheus-ebm SDK + required packages. Silent on Kaggle to avoid log spam.
_install_quiet = '-q' if IS_KAGGLE else ''

for _pkg in ['prometheus-ebm', 'numpy', 'pandas', 'matplotlib', 'scipy']:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', _install_quiet, _pkg],
        check=False, stdout=subprocess.DEVNULL if IS_KAGGLE else None
    )

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── kbench (Kaggle-only) ──────────────────────────────────────────────────────
# kaggle_benchmarks is only available inside a Kaggle kernel.
# All other environments use the direct API loop in C09 instead.
KAGGLE_KBENCH_AVAILABLE = False
kbench = None

if IS_KAGGLE:
    try:
        import kaggle_benchmarks as kbench
        KAGGLE_KBENCH_AVAILABLE = True
        print(f'kaggle-benchmarks v{kbench.__version__}')
    except ImportError:
        print('WARNING: kaggle_benchmarks not found. Falling back to direct API loop.')
else:
    print(f'Running outside Kaggle ({PLATFORM}) — will use direct API evaluation loop in C09.')

# ── Required Input Files Manifest ────────────────────────────────────────────
INPUT_FILES_MANIFEST = {
    'prometheus_200_multimodel_dataset.json': 'STANDARD / EXTENDED',
    'prometheus_1000_dataset.json':           'DEEP_PROBE',
    'probe_ambiguity.json':                   'Epoch-2 (P01) — UNDERDETERMINED',
    'probe_contradictions.json':              'Epoch-2 (P01) — CONTRADICTORY',
}

print('\nInput files:')
_search_roots = [os.getcwd(), '/kaggle/input', '/content']
for fname, desc in INPUT_FILES_MANIFEST.items():
    found = any(os.path.exists(os.path.join(r, fname)) for r in _search_roots) or os.path.exists(fname)
    status = '\u2713' if found else 'o'
    print(f'  [{status}] {fname:48s}  [{desc}]')

print(f'\nPrometheus EBM SDK version:', end=' ')
try:
    import prometheus_ebm as _pebm
    print(_pebm.__version__)
except ImportError:
    print('not installed — using notebook-embedded scoring engine')

In [ ]:
# [C04] Run Configuration
import os
import hashlib
import time
import re
import numpy as np
import pandas as pd


def stable_int_seed(seed_text, bits=31):
    """Deterministic seed helper (stable across Python sessions)."""
    digest = hashlib.sha256(str(seed_text).encode('utf-8')).hexdigest()
    return int(digest[:16], 16) % (2 ** bits)


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BENCHMARK MODE — change this to switch between run profiles            ║
# ║  STANDARD   : 200-item multi-model, 2 seeds                             ║
# ║  EXTENDED   : 200-item multi-model, 3 seeds, higher stress              ║
# ║  DEEP_PROBE : 1000-item single-model deep diagnostic                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝
BENCHMARK_MODE = 'DEEP_PROBE'
BENCHMARK_MODE = str(BENCHMARK_MODE).strip().upper()
if BENCHMARK_MODE not in {'STANDARD', 'EXTENDED', 'DEEP_PROBE'}:
    raise RuntimeError(f'Unsupported BENCHMARK_MODE: {BENCHMARK_MODE}')

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  EXECUTION MODE                                                         ║
# ║  kaggle             → Kaggle-hosted model inference via kbench          ║
# ║  api                → External provider (OpenRouter, OpenAI, Anthropic) ║
# ║  offline_validation → No model calls. Synthetic pipeline check          ║
# ╚══════════════════════════════════════════════════════════════════════════╝
EXECUTION_MODE = 'kaggle'
EXECUTION_MODE = str(os.getenv('PROMETHEUS_EXECUTION_MODE', EXECUTION_MODE)).strip().lower()
if EXECUTION_MODE not in {'kaggle', 'api', 'offline_validation'}:
    raise RuntimeError(f"Unsupported EXECUTION_MODE '{EXECUTION_MODE}'.")

# ── API Provider (only used when EXECUTION_MODE == 'api') ─────────────────
API_PROVIDER = 'openrouter'   # openrouter | openai | anthropic
API_PROVIDER = str(os.getenv('PROMETHEUS_API_PROVIDER', API_PROVIDER)).strip().lower()

MODEL_PROVIDER = API_PROVIDER if EXECUTION_MODE == 'api' else 'kaggle'
RUN_WITHOUT_MODELS = bool(EXECUTION_MODE == 'offline_validation')

# Explicit credentials (environment vars override)
MODEL_API_KEY     = ''
MODEL_API_BASE_URL = ''

# External-provider target models (used only when EXECUTION_MODE == 'api').
# For kaggle/offline_validation, use MULTI_MODEL_INDICES / DEEP_PROBE_MODEL below.
CUSTOM_TARGET_MODELS  = []
CUSTOM_DEEP_PROBE_MODEL = ''


def _resolve_api_key(provider, explicit_key):
    if str(explicit_key or '').strip():
        return str(explicit_key).strip()
    env_candidates = {
        'openrouter': ['PROMETHEUS_API_KEY', 'OPENROUTER_API_KEY'],
        'openai':     ['PROMETHEUS_API_KEY', 'OPENAI_API_KEY'],
        'anthropic':  ['PROMETHEUS_API_KEY', 'ANTHROPIC_API_KEY'],
    }
    for name in env_candidates.get(provider, ['PROMETHEUS_API_KEY']):
        v = os.getenv(name, '')
        if str(v).strip():
            return str(v).strip()
    return ''


MODEL_API_BASE_URL = str(os.getenv('PROMETHEUS_API_BASE_URL', MODEL_API_BASE_URL)).strip()
if EXECUTION_MODE == 'api':
    MODEL_API_KEY = _resolve_api_key(MODEL_PROVIDER, MODEL_API_KEY)
    if not MODEL_API_KEY:
        raise RuntimeError(
            f"EXECUTION_MODE=api with provider '{MODEL_PROVIDER}' requires an API key. "
            'Set PROMETHEUS_API_KEY or fill MODEL_API_KEY above.'
        )
else:
    MODEL_API_KEY = ''

DRY_RUN = False
EPOCH   = 'PROMETHEUS-Epoch-1'
SEED    = 'prometheus-2026'


def _build_mode_profile(mode_name):
    if mode_name == 'EXTENDED':
        return {
            'mode': mode_name, 'run_scope': 'multi', 'pairwise_required': True,
            'dataset_file': 'prometheus_200_multimodel_dataset.json',
            'min_seeds_required': 3,
            'epoch1_seeds':   [f'{SEED}-s1', f'{SEED}-s2', f'{SEED}-s3'],
            'probe_seeds':    [f'{SEED}-p1', f'{SEED}-p2', f'{SEED}-p3'],
            'rg_bootstrap_iterations': 3000, 'kaggle_budget_mode': True,
            'use_llm_judge': False, 'model_call_retries': 2, 'judge_call_retries': 0,
            'model_timeout_seconds': 14400, 'multistage_sample_n': 20,
            'judge_sensitivity_threshold': 0.25, 'multistage_model_strategy': 'all',
            'multistage_max_models': 5,
        }
    if mode_name == 'DEEP_PROBE':
        return {
            'mode': mode_name, 'run_scope': 'solo', 'pairwise_required': False,
            'dataset_file': 'prometheus_1000_dataset.json',
            'min_seeds_required': 3,
            'epoch1_seeds':   [f'{SEED}-deep-s1', f'{SEED}-deep-s2', f'{SEED}-deep-s3'],
            'probe_seeds':    [f'{SEED}-deep-p1', f'{SEED}-deep-p2', f'{SEED}-deep-p3'],
            'rg_bootstrap_iterations': 2000, 'kaggle_budget_mode': False,
            'use_llm_judge': True, 'model_call_retries': 1, 'judge_call_retries': 1,
            'model_timeout_seconds': 36000, 'multistage_sample_n': 20,
            'judge_sensitivity_threshold': 0.20, 'multistage_model_strategy': 'single_model',
            'multistage_max_models': 1,
        }
    return {
        'mode': 'STANDARD', 'run_scope': 'multi', 'pairwise_required': True,
        'dataset_file': 'prometheus_200_multimodel_dataset.json',
        'min_seeds_required': 2,
        'epoch1_seeds':   [f'{SEED}-s1', f'{SEED}-s2'],
        'probe_seeds':    [f'{SEED}-p1', f'{SEED}-p2'],
        'rg_bootstrap_iterations': 2000, 'kaggle_budget_mode': True,
        'use_llm_judge': False, 'model_call_retries': 1, 'judge_call_retries': 0,
        'model_timeout_seconds': 10800, 'multistage_sample_n': 20,
        'judge_sensitivity_threshold': 0.25, 'multistage_model_strategy': 'top_bottom',
        'multistage_max_models': 5,
    }


ACTIVE_PROFILE    = _build_mode_profile(BENCHMARK_MODE)
RUN_SCOPE         = str(ACTIVE_PROFILE['run_scope']).strip().lower()
PAIRWISE_REQUIRED = bool(ACTIVE_PROFILE['pairwise_required'])

SESSION_START_TIME   = time.time()
KAGGLE_LIMIT_SECONDS = 43200
TIME_RESERVE_SECONDS = 1800 if BENCHMARK_MODE == 'DEEP_PROBE' else 3600


def time_remaining():
    return KAGGLE_LIMIT_SECONDS - (time.time() - SESSION_START_TIME) - TIME_RESERVE_SECONDS


def time_ok():
    return time_remaining() > 0


RESEARCH_GRADE_V1         = True
RUN_RESEARCH_GRADE_BLOCKS  = True
RG_RESAMPLE_ONLY           = True

MIN_SEEDS_REQUIRED   = int(ACTIVE_PROFILE['min_seeds_required'])
EPOCH1_SEEDS         = list(ACTIVE_PROFILE['epoch1_seeds'])
PROBE_SEEDS          = list(ACTIVE_PROFILE['probe_seeds'])
EPOCH2_SEEDS         = list(PROBE_SEEDS)
RG_BOOTSTRAP_ITERATIONS = int(ACTIVE_PROFILE['rg_bootstrap_iterations'])
KAGGLE_BUDGET_MODE   = bool(ACTIVE_PROFILE['kaggle_budget_mode'])
USE_LLM_JUDGE_IN_TASK = bool(ACTIVE_PROFILE['use_llm_judge'])
MODEL_CALL_RETRIES   = int(ACTIVE_PROFILE['model_call_retries'])
JUDGE_CALL_RETRIES   = int(ACTIVE_PROFILE['judge_call_retries'])
MODEL_TIMEOUT_SECONDS = int(ACTIVE_PROFILE['model_timeout_seconds'])
MULTISTAGE_SAMPLE_N  = 20
MULTISTAGE_MODEL_STRATEGY = str(ACTIVE_PROFILE['multistage_model_strategy'])
MULTISTAGE_MAX_MODELS     = int(ACTIVE_PROFILE['multistage_max_models'])

RESUME_FROM_CHECKPOINT = True
EPOCH1_CHECKPOINT_DIR  = (
    '/kaggle/working/epoch1_model_checkpoints'
    if os.path.isdir('/kaggle/working')
    else 'epoch1_model_checkpoints'
)

SKIP_MODELS = []
USE_INDEPENDENT_RUNTIME_JUDGE     = bool(USE_LLM_JUDGE_IN_TASK)
INDEPENDENT_JUDGE_SAMPLE_MAX      = 60
JUDGE_SENSITIVITY_MAX_DISAGREEMENT = float(ACTIVE_PROFILE['judge_sensitivity_threshold'])
PAIRWISE_PERMUTATION_ROUNDS       = 1000
DETERMINATE_KEYWORD_REQUIRED_CAP  = 8
AGI_METACOG_TARGET_SCORE          = 0.85
FINAL_OUTPUT_BASENAME             = 'Final_Output_main'

CUSTOM_DATASET_FILE = ''
DATASET_FILE = str(CUSTOM_DATASET_FILE).strip() or str(ACTIVE_PROFILE['dataset_file'])

if BENCHMARK_MODE == 'EXTENDED':
    DECISION_STRESS_RATIO = 0.40; CLARITY_STRESS_RATIO = 0.20
elif BENCHMARK_MODE == 'DEEP_PROBE':
    DECISION_STRESS_RATIO = 0.30; CLARITY_STRESS_RATIO = 0.15
else:
    DECISION_STRESS_RATIO = 0.25; CLARITY_STRESS_RATIO = 0.10

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  MODEL CATALOG — 1-based index. Identical to V5 notebook.               ║
# ║  To change models: edit MULTI_MODEL_INDICES or DEEP_PROBE_MODEL below.  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
KAGGLE_MODEL_CATALOG = [
    'anthropic/claude-haiku-4-5@20251001',        # 1
    'anthropic/claude-opus-4-1@20250805',          # 2
    'anthropic/claude-opus-4-5@20251101',          # 3
    'anthropic/claude-opus-4-6@default',           # 4
    'anthropic/claude-opus-4-7@default',           # 5
    'anthropic/claude-sonnet-4-5@20250929',        # 6
    'anthropic/claude-sonnet-4-6@default',         # 7
    'anthropic/claude-sonnet-4@20250514',          # 8
    'deepseek-ai/deepseek-r1-0528',                # 9
    'deepseek-ai/deepseek-v3.1',                   # 10
    'deepseek-ai/deepseek-v3.2',                   # 11
    'google/gemini-2.0-flash',                     # 12
    'google/gemini-2.0-flash-lite',                # 13
    'google/gemini-2.5-flash',                     # 14
    'google/gemini-2.5-pro',                       # 15
    'google/gemini-3-flash-preview',               # 16
    'google/gemini-3.1-flash-lite-preview',        # 17
    'google/gemini-3.1-pro-preview',               # 18
    'google/gemma-3-12b',                          # 19
    'google/gemma-3-1b',                           # 20
    'google/gemma-3-27b',                          # 21
    'google/gemma-3-4b',                           # 22
    'google/gemma-4-26b-a4b',                      # 23
    'google/gemma-4-31b',                          # 24
    'openai/gpt-5.4-2026-03-05',                   # 25
    'openai/gpt-5.4-mini-2026-03-17',              # 26
    'openai/gpt-5.4-nano-2026-03-17',              # 27
    'openai/gpt-oss-120b',                         # 28
    'openai/gpt-oss-20b',                          # 29
    'qwen/qwen3-235b-a22b-instruct-2507',          # 30
    'qwen/qwen3-coder-480b-a35b-instruct',         # 31
    'qwen/qwen3-next-80b-a3b-instruct',            # 32
    'qwen/qwen3-next-80b-a3b-thinking',            # 33
    'zai/glm-5',                                   # 34
]

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  MODEL SELECTION — edit indices here to change models                   ║
# ║  MULTI_MODEL_INDICES: 1-based indices for STANDARD / EXTENDED           ║
# ║  DEEP_PROBE_MODEL:    1-based index for DEEP_PROBE                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝
MULTI_MODEL_INDICES = [25, 5, 7, 11, 18]   # gpt-5.4, opus-4-7, sonnet-4-6, deepseek-v3.2, gemini-3.1-pro
DEEP_PROBE_MODEL    = 5                     # claude-opus-4-7


def _clean_model_list(values):
    cleaned, seen = [], set()
    for v in list(values or []):
        name = str(v or '').strip()
        if not name or name in seen:
            continue
        cleaned.append(name); seen.add(name)
    return cleaned


configured_custom_models = _clean_model_list(CUSTOM_TARGET_MODELS)

if EXECUTION_MODE in {'kaggle', 'offline_validation'}:
    if BENCHMARK_MODE == 'DEEP_PROBE':
        idx = max(0, min(DEEP_PROBE_MODEL - 1, len(KAGGLE_MODEL_CATALOG) - 1))
        TARGET_MODELS = [KAGGLE_MODEL_CATALOG[idx]]
        print(f'DEEP_PROBE target: #{DEEP_PROBE_MODEL} -> {TARGET_MODELS[0]}')
    else:
        TARGET_MODELS = [
            KAGGLE_MODEL_CATALOG[i - 1]
            for i in MULTI_MODEL_INDICES
            if 1 <= i <= len(KAGGLE_MODEL_CATALOG)
        ]
else:  # api mode
    if BENCHMARK_MODE == 'DEEP_PROBE':
        deep_target = str(CUSTOM_DEEP_PROBE_MODEL or '').strip()
        if deep_target:
            TARGET_MODELS = [deep_target]
        elif configured_custom_models:
            TARGET_MODELS = [configured_custom_models[0]]
        else:
            raise RuntimeError('API DEEP_PROBE requires CUSTOM_DEEP_PROBE_MODEL or CUSTOM_TARGET_MODELS[0].')
    else:
        if len(configured_custom_models) < 2:
            raise RuntimeError('API STANDARD/EXTENDED runs require at least 2 models in CUSTOM_TARGET_MODELS.')
        TARGET_MODELS = list(configured_custom_models)

# V5 fix: judge pool always inherits from evaluation pool.
INDEPENDENT_JUDGE_CANDIDATES = list(TARGET_MODELS)

if len(EPOCH1_SEEDS) < MIN_SEEDS_REQUIRED or len(EPOCH2_SEEDS) < MIN_SEEDS_REQUIRED:
    raise ValueError(f'Research-grade mode requires at least {MIN_SEEDS_REQUIRED} seeds per epoch.')

print('PROMETHEUS-EBM Configuration')
print(f'  Mode: {BENCHMARK_MODE} | Scope: {RUN_SCOPE} | Pairwise: {PAIRWISE_REQUIRED}')
print(f'  Execution: {EXECUTION_MODE} | Provider: {MODEL_PROVIDER} | Platform: {globals().get("PLATFORM", "unknown")}')
print(f'  Models ({len(TARGET_MODELS)}): {TARGET_MODELS}')
print(f'  Dataset: {DATASET_FILE}')
if RUN_WITHOUT_MODELS:
    print('  OFFLINE VALIDATION MODE — no model calls; synthetic responses.')

FORMAT_REINFORCEMENT_MODELS = {
    'deepseek-ai/deepseek-v3.2': (
        '\n\nCRITICAL FORMAT REQUIREMENT: You MUST structure your response '
        'using exactly these five fields, each on its own line:\n'
        'FINAL_ANSWER: <your answer>\n'
        'SOLVABILITY_CLASS: Determinate | Underdetermined | Insufficient | Contradictory\n'
        'CONFIDENCE: <integer 0-100>\n'
        'JUSTIFICATION_TYPE: <deductive | inductive | abductive | ...>\n'
        'REASONING: <your reasoning>'
    ),
}


def _get_format_reinforcement(model_id):
    model_id_lower = str(model_id or '').lower()
    for pattern, suffix in FORMAT_REINFORCEMENT_MODELS.items():
        if pattern.lower() in model_id_lower:
            return suffix
    return ''

In [ ]:
# [C05] Model Resolution
import os
import json
import urllib.request
import urllib.error


def _normalize_target_models(target_models):
    cleaned, seen = [], set()
    for target in list(target_models or []):
        name = str(target or '').strip()
        if not name or name in seen:
            continue
        cleaned.append(name); seen.add(name)
    return cleaned


def _resolve_offline_no_model_mode():
    return str(globals().get('EXECUTION_MODE', 'kaggle')).strip().lower() == 'offline_validation'


def _list_kaggle_model_pool():
    if not globals().get('KAGGLE_KBENCH_AVAILABLE', False) or kbench is None:
        return []
    pool = []
    if hasattr(kbench, 'llms'):
        llms_obj = kbench.llms
        if isinstance(llms_obj, dict):
            for name, obj in llms_obj.items():
                pool.append((str(name), obj))
        else:
            try:
                for obj in llms_obj:
                    name = getattr(obj, 'model', str(obj))
                    pool.append((str(name), obj))
            except Exception:
                pass
    if len(pool) == 0 and hasattr(kbench, 'kaggle') and hasattr(kbench.kaggle, 'load_available_models'):
        try:
            available = kbench.kaggle.load_available_models()
            if isinstance(available, dict):
                for name, obj in available.items():
                    pool.append((str(name), obj))
        except Exception:
            pass
    if len(pool) == 0 and hasattr(kbench, 'kaggle') and hasattr(kbench.kaggle, 'load_model'):
        for target in list(globals().get('TARGET_MODELS', [])):
            try:
                obj = kbench.kaggle.load_model(str(target))
                pool.append((str(target), obj))
            except Exception:
                pass
    dedup, seen = [], set()
    for name, obj in pool:
        key = str(name).strip().lower()
        if not key or key in seen:
            continue
        dedup.append((str(name), obj)); seen.add(key)
    return dedup


def _list_model_pool():
    return _list_kaggle_model_pool()


class ExternalChatModel:
    """Lightweight chat wrapper — kbench-compatible prompt() method."""

    def __init__(self, provider, model, api_key, base_url='', timeout_seconds=120):
        self.provider = str(provider).strip().lower()
        self.model    = str(model).strip()
        self.api_key  = str(api_key).strip()
        self.base_url = str(base_url or '').strip()
        self.timeout_seconds = max(15, int(timeout_seconds))

    def __repr__(self):
        return f'ExternalChatModel(provider={self.provider}, model={self.model})'

    @staticmethod
    def _coerce_text(content):
        if isinstance(content, str):
            return content
        if isinstance(content, list):
            parts = []
            for item in content:
                if isinstance(item, dict):
                    parts.append(str(item.get('text', item.get('content', str(item)))))
                else:
                    parts.append(str(item))
            return '\n'.join([p for p in parts if p]).strip()
        if isinstance(content, dict):
            return str(content.get('text', content.get('content', ''))).strip()
        return str(content or '').strip()

    def _post_json(self, url, payload, headers):
        data = json.dumps(payload).encode('utf-8')
        req  = urllib.request.Request(url=url, data=data, headers=headers, method='POST')
        try:
            with urllib.request.urlopen(req, timeout=self.timeout_seconds) as resp:
                body = resp.read().decode('utf-8', errors='replace')
        except urllib.error.HTTPError as exc:
            body = exc.read().decode('utf-8', errors='replace') if hasattr(exc, 'read') else ''
            raise RuntimeError(f'{self.provider} HTTP {exc.code} for {self.model}: {body[:600]}') from exc
        except urllib.error.URLError as exc:
            raise RuntimeError(f'{self.provider} connection error for {self.model}: {exc}') from exc
        try:
            return json.loads(body)
        except json.JSONDecodeError as exc:
            raise RuntimeError(f'{self.provider} returned non-JSON for {self.model}.') from exc

    def prompt(self, user_text, system=None):
        user_text   = str(user_text or '')
        system_text = str(system or '').strip()

        if self.provider in {'openrouter', 'openai'}:
            messages = []
            if system_text:
                messages.append({'role': 'system', 'content': system_text})
            messages.append({'role': 'user', 'content': user_text})
            default_url = {
                'openrouter': 'https://openrouter.ai/api/v1/chat/completions',
                'openai':     'https://api.openai.com/v1/chat/completions',
            }[self.provider]
            headers = {
                'Authorization': f'Bearer {self.api_key}',
                'Content-Type':  'application/json',
            }
            payload = {'model': self.model, 'messages': messages}
            if self.provider == 'openrouter':
                payload['temperature'] = 0
            response = self._post_json(url=self.base_url or default_url, payload=payload, headers=headers)
            try:
                content = response['choices'][0]['message']['content']
            except Exception as exc:
                raise RuntimeError(f'Unexpected {self.provider} schema for {self.model}.') from exc
            text = self._coerce_text(content)
            if not text:
                raise RuntimeError(f'{self.provider} response was empty for {self.model}.')
            return text

        if self.provider == 'anthropic':
            url = self.base_url or 'https://api.anthropic.com/v1/messages'
            headers = {
                'x-api-key':       self.api_key,
                'anthropic-version': '2023-06-01',
                'Content-Type':    'application/json',
            }
            payload = {
                'model':      self.model,
                'max_tokens': 1024,
                'messages':   [{'role': 'user', 'content': user_text}],
            }
            if system_text:
                payload['system'] = system_text
            response = self._post_json(url=url, payload=payload, headers=headers)
            text = self._coerce_text(response.get('content', []))
            if not text:
                raise RuntimeError(f'anthropic response was empty for {self.model}.')
            return text

        raise RuntimeError(f'Unsupported external provider: {self.provider}')


def _resolve_kaggle_targets(target_models):
    pool     = _list_kaggle_model_pool()
    pool_map = {n.lower(): (n, o) for n, o in pool}
    resolved, missing = [], []
    for target in target_models:
        t = target.lower()
        if t in pool_map:
            resolved.append(pool_map[t]); continue
        candidates = [(n, o) for (n, o) in pool if t in n.lower() or n.lower() in t]
        if candidates:
            resolved.append(sorted(candidates, key=lambda x: len(x[0]))[0])
        else:
            missing.append(target)
    seen, dedup = set(), []
    for n, o in resolved:
        key = str(n).lower()
        if key in seen: continue
        dedup.append((n, o)); seen.add(key)
    return dedup, pool, missing


def resolve_targets(target_models):
    provider           = str(globals().get('MODEL_PROVIDER', 'kaggle')).strip().lower()
    normalized_targets = _normalize_target_models(target_models)
    if not normalized_targets:
        return [], [], []

    if provider == 'kaggle':
        return _resolve_kaggle_targets(normalized_targets)

    api_key    = str(globals().get('MODEL_API_KEY', '')).strip()
    base_url   = str(globals().get('MODEL_API_BASE_URL', '')).strip()
    rtimeout   = int(os.getenv('PROMETHEUS_API_REQUEST_TIMEOUT_SECONDS', '120'))
    if not api_key:
        raise RuntimeError(f'MODEL_PROVIDER={provider} requires MODEL_API_KEY.')
    resolved = [
        (m, ExternalChatModel(
            provider=provider, model=m, api_key=api_key,
            base_url=base_url, timeout_seconds=rtimeout,
        ))
        for m in normalized_targets
    ]
    return resolved, list(resolved), []


provider_name           = str(globals().get('MODEL_PROVIDER', 'kaggle')).strip().lower()
RUN_WITHOUT_MODELS      = bool(globals().get('RUN_WITHOUT_MODELS', _resolve_offline_no_model_mode()))
USING_OFFLINE_SYNTHETIC_RESPONSES = bool(RUN_WITHOUT_MODELS)

if RUN_WITHOUT_MODELS:
    offline_targets = _normalize_target_models(TARGET_MODELS)
    if not offline_targets:
        raise RuntimeError('RUN_WITHOUT_MODELS is enabled but TARGET_MODELS is empty.')
    models_to_run  = [(name, None) for name in offline_targets]
    all_pool       = list(models_to_run)
    missing_targets = []
    print(f'Model provider: {provider_name}')
    print('Offline mode: synthetic responses will be generated in C09.')
else:
    models_to_run, all_pool, missing_targets = resolve_targets(TARGET_MODELS)
    if not all_pool:
        if provider_name == 'kaggle':
            raise RuntimeError('No models loaded from kaggle_benchmarks.')
        raise RuntimeError(f"No models initialized for provider '{provider_name}'.")
    print(f'Model provider: {provider_name}')
    for n, _ in all_pool:
        print(f'  {n}')
    if missing_targets:
        raise RuntimeError(f'Could not resolve {len(missing_targets)} target model(s): {missing_targets}')

expected_target_count = len(_normalize_target_models(TARGET_MODELS))
if len(models_to_run) != expected_target_count:
    raise RuntimeError(f'Expected {expected_target_count} models, resolved {len(models_to_run)}.')

run_scope = str(globals().get('RUN_SCOPE', 'multi')).strip().lower()
if run_scope == 'solo'  and len(models_to_run) != 1:
    raise RuntimeError(f'Solo mode requires exactly 1 resolved model; got {len(models_to_run)}.')
if run_scope == 'multi' and len(models_to_run) < 2:
    raise RuntimeError(f'Multi mode requires at least 2 resolved models; got {len(models_to_run)}.')
if BENCHMARK_MODE == 'DEEP_PROBE' and len(models_to_run) != 1:
    raise RuntimeError('DEEP_PROBE must resolve exactly one model.')

In [ ]:
# [C06] Scoring Engine: Parser, Evaluator, and ECI Scorer
import os
import glob
import re
import random
import hashlib
import importlib.util
from dataclasses import dataclass
from collections import defaultdict

def _find_file(filename):
    candidates = []
    for base in [os.getcwd(), '/kaggle/working', '/kaggle/input', '/content']:
        p = os.path.join(base, filename)
        if os.path.isfile(p):
            candidates.append(p)
    for pat in [f'/kaggle/input/**/{filename}', f'/kaggle/working/**/{filename}',
                f'{os.getcwd()}/**/{filename}']:
        candidates.extend(glob.glob(pat, recursive=True))
    seen, uniq = set(), []
    for c in candidates:
        if c not in seen:
            seen.add(c); uniq.append(c)
    return uniq[0] if uniq else None

def _load_module(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

def _build_fallback_modules():
    @dataclass
    class ParsedResponse:
        problem_id: str; raw_response: str; final_answer: object
        solvability_class_estimate: object; confidence: object
        justification_type: object; reasoning: object
        parse_success: bool; parse_error: object = None

    def parse_response(problem_id, raw_response):
        text = str(raw_response or '').strip()
        def extract(field):
            pat = rf'{field}:\s*(.+?)(?=\n[A-Z_]+:|$)'
            m = re.search(pat, text, re.DOTALL | re.IGNORECASE)
            return m.group(1).strip() if m else None
        final_answer   = extract('FINAL_ANSWER')
        solv_raw       = extract('SOLVABILITY_CLASS')
        conf_raw       = extract('CONFIDENCE')
        just           = extract('JUSTIFICATION_TYPE')
        reason         = extract('REASONING')
        solvability = None
        if solv_raw:
            s = solv_raw.lower()
            if 'under' in s:              solvability = 'Underdetermined'
            elif 'insuff' in s:           solvability = 'Insufficient'
            elif 'contrad' in s:          solvability = 'Contradictory'
            elif 'determin' in s:         solvability = 'Determinate'
        confidence = 0.5
        if conf_raw:
            nums = re.findall(r'\d+\.?\d*', conf_raw)
            if nums:
                v = float(nums[0])
                confidence = max(0.0, min(1.0, v / 100.0 if v > 1.0 else v))
        ok = final_answer is not None and solvability is not None
        return ParsedResponse(
            problem_id=problem_id, raw_response=text, final_answer=final_answer,
            solvability_class_estimate=solvability, confidence=confidence,
            justification_type=just, reasoning=reason, parse_success=ok,
            parse_error=None if ok else 'Missing required fields'
        )

    def evaluate_answer_correctness(model_answer, ground_truth, problem_class,
                                    solvability_estimate, judge_fn=None):
        if problem_class == 'DETERMINATE':
            if model_answer is None or ground_truth in (None, '', 'None'):
                return False, 'missing_answer_or_gt'
            if judge_fn is not None:
                try:
                    judged = judge_fn(model_answer, ground_truth)
                    if isinstance(judged, tuple):
                        return bool(judged[0]), 'judge_evaluation'
                    return bool(judged), 'judge_evaluation'
                except Exception:
                    pass
            def _norm(v):
                s = re.sub(r'[^a-z0-9\s]', ' ', str(v or '').lower())
                return re.sub(r'\s+', ' ', s).strip()
            ans_norm = _norm(model_answer)
            gt_norm  = _norm(ground_truth)
            if not ans_norm: return False, 'empty_answer'
            if ans_norm == gt_norm or ans_norm in gt_norm or gt_norm in ans_norm:
                return True, 'containment'
            ans_t, gt_t = set(ans_norm.split()), set(gt_norm.split())
            if gt_t:
                overlap = len(ans_t & gt_t) / len(gt_t)
                if overlap >= 0.5 and len(ans_t & gt_t) >= max(1, min(3, len(gt_t))):
                    return True, 'token_overlap'
            key_terms = [t for t in gt_norm.split() if len(t) > 4]
            if not key_terms: return False, 'no_key_terms'
            hits = sum(1 for t in key_terms if t in ans_norm)
            required_cap = int(globals().get('DETERMINATE_KEYWORD_REQUIRED_CAP', 8))
            return hits >= max(1, min(required_cap, len(key_terms) // 4)), 'keyword_fallback'
        if problem_class == 'UNDERDETERMINED':
            if solvability_estimate == 'Underdetermined': return True, 'correct_class'
            if model_answer and any(w in str(model_answer).lower()
                                    for w in ['multiple', 'depends', 'ambiguous']):
                return True, 'ambiguity_acknowledged'
            return False, 'failed_underdetermined'
        if problem_class == 'INSUFFICIENT':
            if solvability_estimate == 'Insufficient': return True, 'correct_class'
            if model_answer and any(w in str(model_answer).lower()
                                    for w in ['cannot', 'insufficient', 'not enough', 'missing']):
                return True, 'correct_refusal'
            return False, 'hallucinated_on_insufficient'
        if problem_class == 'CONTRADICTORY':
            if solvability_estimate == 'Contradictory': return True, 'correct_class'
            if model_answer and any(w in str(model_answer).lower()
                                    for w in ['contradict', 'inconsistent', 'impossible', 'conflict']):
                return True, 'contradiction_id'
            return False, 'failed_contradiction'
        return False, 'unknown_class'

    class ECIScorer:
        WEIGHTS = {'SDA': 0.30, 'CA': 0.25, 'RP': 0.20, 'ECE': 0.15, 'HSS': 0.10}
        def __init__(self, model_name, epoch):
            self.model_name = model_name; self.epoch = epoch
        def score(self, problems, raw_responses, judge_fn=None):
            parsed = [parse_response(p['problem_id'], r) for p, r in zip(problems, raw_responses)]
            rows = []
            for prob, resp in zip(problems, parsed):
                is_correct, _ = evaluate_answer_correctness(
                    resp.final_answer, prob.get('ground_truth_answer'),
                    prob.get('problem_class', ''), resp.solvability_class_estimate, judge_fn=judge_fn)
                solv_ok = resp.solvability_class_estimate == prob.get('correct_solvability_class')
                conf    = float(resp.confidence if resp.confidence is not None else 0.5)
                is_refusal   = any(w in str(resp.final_answer or '').lower()
                                   for w in ['refusal','cannot','insufficient','contradict'])
                should_refuse = prob.get('problem_class') in ['INSUFFICIENT','CONTRADICTORY']
                rows.append({'correct': bool(is_correct), 'solv_ok': bool(solv_ok), 'conf': conf,
                             'is_refusal': is_refusal, 'should_refuse': should_refuse,
                             'problem_class': prob.get('problem_class',''),
                             'domain': prob.get('domain',''), 'bf': int(prob.get('branching_factor',1))})
            n   = max(1, len(rows))
            sda = sum(1 for r in rows if r['solv_ok']) / n
            determ = [r for r, p in zip(rows, problems) if p.get('problem_class') == 'DETERMINATE']
            ca     = (sum(1 for r in determ if r['correct']) / len(determ)) if determ else 0.0
            refusals = [r for r in rows if r['is_refusal']]
            rp       = (sum(1 for r in refusals if r['should_refuse']) / len(refusals)) if refusals else 1.0
            ece      = sum(abs((1.0 if r['correct'] else 0.0) - r['conf']) for r in rows) / n
            impossible = [r for r, p in zip(rows, problems)
                          if p.get('problem_class') in ['INSUFFICIENT','CONTRADICTORY']]
            hss = (sum(1.0 for r in impossible if (not r['correct']) and r['conf'] > 0.6) /
                   len(impossible)) if impossible else 0.0
            eci = (self.WEIGHTS['SDA'] * sda + self.WEIGHTS['CA'] * ca +
                   self.WEIGHTS['RP'] * rp + self.WEIGHTS['ECE'] * (1.0 - ece) +
                   self.WEIGHTS['HSS'] * (1.0 - hss))
            per_class, per_domain, per_bf = {}, {}, {}
            c_acc, d_acc, b_acc = defaultdict(list), defaultdict(list), defaultdict(list)
            for r, p in zip(rows, problems):
                c_acc[p.get('problem_class','')].append(1.0 if r['correct'] else 0.0)
                d_acc[p.get('domain','')].append(1.0 if r['correct'] else 0.0)
                b_acc[int(p.get('branching_factor',1))].append(1.0 if r['correct'] else 0.0)
            for k, vs in c_acc.items(): per_class[k] = float(sum(vs)/len(vs))
            for k, vs in d_acc.items():  per_domain[k] = float(sum(vs)/len(vs))
            for k, vs in b_acc.items():  per_bf[k]    = float(sum(vs)/len(vs))
            overconf = sum(1 for r in rows if (not r['correct']) and r['conf'] > 0.8)
            cbu    = sum(1 for r in rows if r['correct'] and r['conf'] < 0.5)
            hallu  = sum(1 for r in rows if r['should_refuse'] and (not r['is_refusal']) and r['conf'] > 0.6)
            corr_ref = sum(1 for r in rows if r['should_refuse'] and r['is_refusal'])

            class _ECIResult:
                pass
            res = _ECIResult()
            res.model_name = self.model_name; res.epoch = self.epoch
            res.n_problems = len(rows); res.n_parsed = len(rows)
            res.solvability_detection_accuracy = sda; res.conditional_accuracy = ca
            res.refusal_precision = rp; res.expected_calibration_error = ece
            res.hallucination_severity_score = hss; res.eci_score = eci
            res.per_class_accuracy = per_class; res.per_domain_accuracy = per_domain
            res.per_bf_accuracy = per_bf; res.calibration_bins = []
            res.overconfident_wrong = overconf; res.correct_but_uncertain = cbu
            res.hallucinated_on_impossible = hallu; res.correct_refusals = corr_ref
            res.summary = lambda: (
                f'ECI={eci:.4f} | SDA={sda:.4f} | CA={ca:.4f} | '
                f'RP={rp:.4f} | ECE={ece:.4f} | HSS={hss:.4f}'
            )
            return res

    class BenchmarkBuilder:
        def __init__(self, epoch='PROMETHEUS-Epoch-1', seed='prometheus-2026'):
            self.epoch = epoch; self.rng = random.Random(str(seed))
            self.domains = ['medical','financial','legal','environmental','social']
            self.classes = ['DETERMINATE','UNDERDETERMINED','INSUFFICIENT','CONTRADICTORY']
        def _make_problem_id(self, i, cls, dom, bf):
            raw = f'{self.epoch}-{i}-{cls}-{dom}-bf{bf}'
            return hashlib.sha256(raw.encode('utf-8')).hexdigest()[:12].upper()
        def build(self, n_problems=500):
            problems = []
            for i in range(n_problems):
                cls = self.classes[i % len(self.classes)]
                dom = self.domains[(i // len(self.classes)) % len(self.domains)]
                bf  = 1 + (i % 5)
                if   cls == 'DETERMINATE':   gt = f'Deterministic answer token {i%17}'; solv = 'Determinate'; q = f'In {dom}, compute the unique outcome for case {i} with bf {bf}.'
                elif cls == 'UNDERDETERMINED': gt = None; solv = 'Underdetermined'; q = f'In {dom}, case {i} has multiple equally plausible outcomes. What is the answer?'
                elif cls == 'INSUFFICIENT':  gt = None; solv = 'Insufficient'; q = f'In {dom}, key facts are missing for case {i}. Provide the definitive answer.'
                else:                         gt = None; solv = 'Contradictory'; q = f'In {dom}, premises conflict for case {i}. What is the final answer?'
                problems.append({
                    'problem_id': self._make_problem_id(i, cls, dom, bf), 'epoch': self.epoch,
                    'domain': dom, 'problem_class': cls, 'branching_factor': bf,
                    'question': q, 'ground_truth_answer': gt, 'correct_solvability_class': solv,
                    'metadata': {'fallback_generator': True}
                })
            return problems
        def build_model_prompts(self, problems):
            out = []
            system = ('You are solving PROMETHEUS-EBM tasks. Always answer using exact fields:\n'
                      'FINAL_ANSWER, SOLVABILITY_CLASS, CONFIDENCE, JUSTIFICATION_TYPE, REASONING.')
            for p in problems:
                user = (f"Problem ID: {p['problem_id']}\nDomain: {p['domain']}\nQuestion: {p['question']}\n\n"
                        'Return exactly:\nFINAL_ANSWER: ...\nSOLVABILITY_CLASS: ...\nCONFIDENCE: <0-100>\nJUSTIFICATION_TYPE: ...\nREASONING: ...')
                out.append({'problem_id': p['problem_id'], 'system': system, 'user': user,
                            'ground_truth_answer': p['ground_truth_answer'],
                            'correct_solvability_class': p['correct_solvability_class'],
                            'problem_class': p['problem_class'], 'domain': p['domain'],
                            'branching_factor': p['branching_factor']})
            return out

    return BenchmarkBuilder, parse_response, evaluate_answer_correctness, ECIScorer

pg_path = _find_file('problem_generator.py')
se_path = _find_file('scoring_engine.py')
if pg_path and se_path:
    problem_generator_mod = _load_module('problem_generator_mod', pg_path)
    scoring_engine_mod    = _load_module('scoring_engine_mod',    se_path)
    BenchmarkBuilder          = problem_generator_mod.BenchmarkBuilder
    parse_response            = scoring_engine_mod.parse_response
    evaluate_answer_correctness = scoring_engine_mod.evaluate_answer_correctness
    ECIScorer                 = scoring_engine_mod.ECIScorer
    print('External scoring engine loaded.')
else:
    BenchmarkBuilder, parse_response, evaluate_answer_correctness, ECIScorer = _build_fallback_modules()
    print('Using notebook-embedded scoring engine.')

# Brier Score Decomposition
def brier_score_decomposition(confidences, correctness_flags, n_bins=10):
    confs   = np.array(confidences, dtype=float)
    correct = np.array(correctness_flags, dtype=float)
    if len(confs) == 0:
        return {'brier': float('nan'),'reliability':float('nan'),'resolution':float('nan'),'uncertainty':float('nan')}
    brier      = float(np.mean((confs - correct) ** 2))
    base_rate  = float(np.mean(correct))
    uncertainty = base_rate * (1 - base_rate)
    bin_edges   = np.linspace(0, 1, n_bins + 1)
    reliability = resolution = 0.0
    for k in range(n_bins):
        mask = (confs >= bin_edges[k]) & (confs < bin_edges[k+1])
        if k == n_bins-1: mask |= (confs == bin_edges[k+1])
        n_k = int(np.sum(mask))
        if n_k == 0: continue
        avg_conf_k = float(np.mean(confs[mask])); avg_corr_k = float(np.mean(correct[mask]))
        reliability += n_k * (avg_corr_k - avg_conf_k) ** 2
        resolution  += n_k * (avg_corr_k - base_rate) ** 2
    n_total = len(confs)
    return {'brier': round(brier,6), 'reliability': round(reliability/n_total,6),
            'resolution': round(resolution/n_total,6), 'uncertainty': round(uncertainty,6)}

# Type-2 D-Prime
def type2_d_prime(confidences, correctness_flags, threshold=0.7):
    from scipy.stats import norm
    confs   = np.array(confidences, dtype=float)
    correct = np.array(correctness_flags, dtype=bool)
    confident = confs >= threshold
    n_correct = int(np.sum(correct)); n_incorrect = int(np.sum(~correct))
    if n_correct == 0 or n_incorrect == 0:
        return {'d_prime': float('nan'),'hit_rate':float('nan'),'false_alarm_rate':float('nan'),'threshold':threshold}
    hit_rate = float(np.sum(confident & correct)) / n_correct
    fa_rate  = float(np.sum(confident & ~correct)) / n_incorrect
    hit_adj  = (hit_rate * n_correct + 0.5) / (n_correct + 1)
    fa_adj   = (fa_rate  * n_incorrect + 0.5) / (n_incorrect + 1)
    d_prime  = float(norm.ppf(hit_adj) - norm.ppf(fa_adj))
    return {'d_prime': round(d_prime,4), 'hit_rate': round(hit_rate,4),
            'false_alarm_rate': round(fa_rate,4), 'threshold': threshold}

print('Scoring engine initialized: ECI, HGI, Brier decomposition, Type-2 D-Prime.')

In [ ]:
# [C07] Dataset Loading and Stress Augmentation
import os, json, glob
import pandas as pd
import numpy as np

def find_dataset_file(filename):
    search_paths = [
        filename, os.path.join(os.getcwd(), filename),
        os.path.join('/kaggle/input', filename), os.path.join('/kaggle/working', filename),
        os.path.join('/content', filename),
    ]
    for path in search_paths:
        if os.path.isfile(path): return path
    for root_dir in ['/kaggle/input', '/content']:
        if os.path.isdir(root_dir):
            for root, dirs, files in os.walk(root_dir):
                if filename in files: return os.path.join(root, filename)
    for pat in [f'/kaggle/input/**/{filename}', f'/content/**/{filename}']:
        matches = glob.glob(pat, recursive=True)
        if matches: return matches[0]
    return None

dataset_path = find_dataset_file(DATASET_FILE)
if dataset_path:
    print(f'Dataset source: {dataset_path}')
    with open(dataset_path, 'r', encoding='utf-8') as f:
        raw_problems = json.load(f)
    print(f'Loaded {len(raw_problems)} problems')
else:
    exec_mode = str(globals().get('EXECUTION_MODE','kaggle')).strip().lower()
    if exec_mode != 'offline_validation':
        raise FileNotFoundError(
            f"Dataset '{DATASET_FILE}' not found. Real runs require a real dataset file. "
            'For pipeline testing, set EXECUTION_MODE = "offline_validation".'
        )
    print(f"Dataset not found — using synthetic fallback (offline_validation only).")
    builder = BenchmarkBuilder(epoch=EPOCH, seed=SEED)
    count = 24 if DRY_RUN else 500
    raw_problems = builder.build(n_problems=count)
    for p in raw_problems:
        if 'user' not in p and 'question' in p: p['user'] = p['question']
        if 'system' not in p:
            p['system'] = ('You are a rigorous analytical reasoning system. '
                           'Respond with FINAL_ANSWER, SOLVABILITY_CLASS, CONFIDENCE, '
                           'JUSTIFICATION_TYPE, and REASONING.')
    print(f'Generated {len(raw_problems)} synthetic problems')

if DRY_RUN and len(raw_problems) > 24:
    rng = np.random.default_rng(42)
    indices = rng.choice(len(raw_problems), size=24, replace=False)
    raw_problems = [raw_problems[i] for i in sorted(indices)]
    print(f'DRY_RUN: subsampled to {len(raw_problems)} problems')

BASE_RAW_PROBLEMS = [dict(p) for p in raw_problems]

prompts = []
for p in raw_problems:
    prompts.append({
        'problem_id': p['problem_id'], 'system': p.get('system',''),
        'user': p.get('user', p.get('question','')),
        'ground_truth_answer': p.get('ground_truth_answer'),
        'correct_solvability_class': p.get('correct_solvability_class'),
        'problem_class': p.get('problem_class'), 'domain': p.get('domain'),
        'branching_factor': p.get('branching_factor', 2), 'rigor_mode': p.get('rigor_mode','base'),
    })

augmented = list(prompts)
seed_int  = stable_int_seed(SEED) if 'stable_int_seed' in globals() else 42
rng       = np.random.default_rng(seed_int)

for p in prompts:
    roll = rng.random()
    if roll < DECISION_STRESS_RATIO:
        aug = dict(p); aug['problem_id'] = p['problem_id'] + '-DS'
        aug['user'] = (p['user'] + '\n\nAdditional instruction: '
                       'Before finalizing your answer, explicitly test at least one plausible '
                       'alternative interpretation and reject it with evidence if unsupported.')
        aug['rigor_mode'] = 'decision_stress'; augmented.append(aug)
    elif roll < DECISION_STRESS_RATIO + CLARITY_STRESS_RATIO:
        aug = dict(p); aug['problem_id'] = p['problem_id'] + '-CS'
        aug['user'] = (p['user'] + '\n\nAdditional instruction: '
                       'Maintain strict adherence to the response schema. Avoid any speculative '
                       'claims not directly supported by the information given.')
        aug['rigor_mode'] = 'clarity_stress'; augmented.append(aug)

prompts = augmented
df = pd.DataFrame(prompts)

required_cols = ['problem_id','system','user','ground_truth_answer',
                 'correct_solvability_class','problem_class','domain','branching_factor','rigor_mode']
missing = [c for c in required_cols if c not in df.columns]
assert len(missing) == 0, f'Missing columns: {missing}'
assert df['problem_id'].duplicated().sum() == 0, 'Duplicate problem_ids found'

print(f'Dataset ready: {len(df)} rows, {len(df.columns)} columns')
print('Class distribution:'); print(df['problem_class'].value_counts().to_string())

In [ ]:
# [C08] Benchmark Task Definition — Platform-Adaptive (No kbench.task Required)
# 
# prometheus_ebm_task() is a plain callable — no decorator, no kbench dependency.
# On Kaggle with kbench available, we optionally wrap it as a @kbench.task.
# On Colab / local / any other env, the direct evaluation loop in C09 calls it directly.

import time as _t08

def safe_prompt(llm, user_text, system_text=None, retries=2):
    last_err = None
    for _ in range(retries + 1):
        try:
            if system_text:
                try:    return llm.prompt(user_text, system=system_text)
                except TypeError: return llm.prompt(f'{system_text}\n\n{user_text}')
            return llm.prompt(user_text)
        except Exception as e:
            last_err = e; _t08.sleep(5)
    err_type = type(last_err).__name__ if last_err is not None else 'UnknownError'
    return (
        'FINAL_ANSWER: REFUSAL\nSOLVABILITY_CLASS: Insufficient\nCONFIDENCE: 5\n'
        f'JUSTIFICATION_TYPE: Refusal\nREASONING: Model API failure ({err_type}).'
    )


def make_judge(llm):
    def judge_fn(model_answer, ground_truth):
        judge_prompt = (
            'Is the following model answer correct given the ground truth?\n\n'
            f'Ground truth: {ground_truth}\nModel answer: {model_answer}\n\nReply: CORRECT or INCORRECT'
        )
        try:
            resp = safe_prompt(llm, judge_prompt, system_text=None, retries=1)
            text = str(resp).upper()
            if 'INCORRECT' in text: return False
            if 'CORRECT'   in text: return True
        except Exception:
            pass
        if model_answer is None or ground_truth is None: return False
        gt = str(ground_truth).lower(); ans = str(model_answer).lower()
        key_terms = [t for t in gt.split() if len(t) > 4]
        return sum(1 for t in key_terms if t in ans) >= max(1, len(key_terms) // 3)
    return judge_fn


def _pick_runtime_judge_llm(eval_llm):
    if not bool(globals().get('USE_INDEPENDENT_RUNTIME_JUDGE', False)):
        return eval_llm
    model_pool = [obj for _, obj in globals().get('models_to_run', [])]
    for cand in model_pool:
        if cand is not eval_llm: return cand
    return eval_llm


def prometheus_ebm_task(
    llm, problem_id, system, user, ground_truth_answer,
    correct_solvability_class, problem_class, domain,
    branching_factor, rigor_mode='base', **kwargs
):
    """Core benchmark task — callable directly, no kbench dependency."""
    _ = kwargs
    try:
        judge_llm = _pick_runtime_judge_llm(llm)
        judge     = make_judge(judge_llm)
        response  = safe_prompt(llm, user_text=user, system_text=system, retries=2)
        parsed    = parse_response(problem_id, response)
        gt = ground_truth_answer if ground_truth_answer not in ('None', None, '') else None
        is_correct, eval_method = evaluate_answer_correctness(
            parsed.final_answer, gt, problem_class, parsed.solvability_class_estimate, judge_fn=judge
        )
        solvability_correct = parsed.solvability_class_estimate == correct_solvability_class
        conf         = parsed.confidence if parsed.confidence is not None else 0.5
        is_impossible  = problem_class in ['INSUFFICIENT','CONTRADICTORY','UNDERDETERMINED']
        is_hallucination = is_impossible and (not is_correct) and conf > 0.6
        if   is_correct and solvability_correct:      item_score = 1.0
        elif is_correct and not solvability_correct:  item_score = 0.7
        elif (not is_correct) and solvability_correct: item_score = 0.4
        elif is_hallucination:                         item_score = -0.3
        else:                                          item_score = 0.0
        if (not is_correct) and conf > 0.8: item_score -= 0.2
        normalized = max(0.0, min(1.0, (item_score + 0.3) / 1.3))
        return {
            'response': response, 'is_correct': is_correct, 'normalized_score': normalized,
            'eval_method': eval_method, 'solvability_correct': solvability_correct,
            'confidence': conf, 'item_score': item_score,
        }
    except Exception as e:
        fallback = (
            'FINAL_ANSWER: REFUSAL\nSOLVABILITY_CLASS: Insufficient\nCONFIDENCE: 5\n'
            f'JUSTIFICATION_TYPE: Refusal\nREASONING: Internal benchmark error ({type(e).__name__}).'
        )
        return {
            'response': fallback, 'is_correct': False, 'normalized_score': 0.0,
            'eval_method': 'exception', 'solvability_correct': False,
            'confidence': 0.05, 'item_score': 0.0,
        }


# ── Optional kbench.task registration (Kaggle only) ──────────────────────────
# When running inside a Kaggle kernel with kbench, we register prometheus_ebm_task
# as a @kbench.task so that kbench.evaluate() can drive the evaluation loop in C09.
# Outside Kaggle, C09 calls prometheus_ebm_task() directly — no kbench needed.
prometheus_ebm = prometheus_ebm_task  # default: plain callable

if globals().get('KAGGLE_KBENCH_AVAILABLE', False) and kbench is not None:
    try:
        @kbench.task(name='prometheus_ebm_v5')
        def _kbench_wrapped_task(llm, problem_id, system, user, ground_truth_answer,
                                  correct_solvability_class, problem_class, domain,
                                  branching_factor, rigor_mode='base', **kwargs):
            result = prometheus_ebm_task(
                llm, problem_id, system, user, ground_truth_answer,
                correct_solvability_class, problem_class, domain, branching_factor, rigor_mode, **kwargs
            )
            kbench.assertions.assert_true(
                result['normalized_score'] > 0.5,
                expectation=(
                    f'EBM | class={problem_class} domain={domain} bf={branching_factor} '
                    f'rigor={rigor_mode} correct={result["is_correct"]}({result["eval_method"]}) '
                    f'solvability_ok={result["solvability_correct"]} '
                    f'conf={result["confidence"]:.2f} score={result["item_score"]:.2f}'
                ),
            )
            return None
        prometheus_ebm = _kbench_wrapped_task
        print('Benchmark task registered with kbench (Kaggle mode).')
    except Exception as e:
        print(f'kbench.task registration failed (non-fatal): {e}')
        print('Falling back to direct evaluation mode.')
else:
    print('Benchmark task initialized (direct mode — Colab/local/offline).')

In [ ]:
# [C09] Multi-Model Evaluation Loop — Platform-Adaptive
# Three evaluation paths:
#   Path A: offline_validation  → synthetic responses (no model calls)
#   Path B: Kaggle + kbench     → kbench.evaluate() drives the loop
#   Path C: api / non-Kaggle    → direct prometheus_ebm_task() call
import os
import re
import time as _time
import numpy as np
import pandas as pd

run_outputs   = {}
failed_models = []


def _textify(content):
    if content is None: return ''
    if isinstance(content, str): return content
    if isinstance(content, list):
        parts = []
        for item in content:
            parts.append(str(item.get('text','') if isinstance(item, dict) else str(item)))
        return '\n'.join(parts)
    if isinstance(content, dict): return str(content.get('text', content))
    return str(content)


def run_to_response_text(run):
    try:
        chat    = getattr(run, 'chat', None)
        history = getattr(chat, 'history', None) or []
        for msg in reversed(history):
            text = _textify(getattr(msg, 'content', None))
            if 'FINAL_ANSWER:' in text and 'SOLVABILITY_CLASS:' in text: return text
        best = ''
        for msg in history:
            text = _textify(getattr(msg, 'content', None))
            if len(text) > len(best): best = text
        return best
    except Exception: return ''


def _model_slug(name):
    return re.sub(r'[^A-Za-z0-9._-]+', '_', str(name))[:120]


def _normalize_solvability_label(problem_class):
    mapping = {'DETERMINATE':'Determinate','UNDERDETERMINED':'Underdetermined',
               'INSUFFICIENT':'Insufficient','CONTRADICTORY':'Contradictory'}
    return mapping.get(str(problem_class or '').strip().upper(), 'Determinate')


def _offline_model_quality(model_name):
    local_seed = stable_int_seed(f'offline-quality::{model_name}')
    rng = np.random.default_rng(local_seed)
    return float(0.62 + 0.28 * rng.random())


def _build_offline_response(problem_row, model_name):
    problem_id = str(problem_row.get('problem_id','unknown'))
    true_label = _normalize_solvability_label(problem_row.get('problem_class'))
    class_labels = ['Determinate','Underdetermined','Insufficient','Contradictory']
    local_seed = stable_int_seed(f'offline::{model_name}::{problem_id}')
    rng = np.random.default_rng(local_seed)
    quality = _offline_model_quality(model_name)
    class_is_correct = bool(rng.random() < min(0.95, max(0.5, quality)))
    predicted_label  = true_label if class_is_correct else str(rng.choice([c for c in class_labels if c != true_label]))
    ground_truth = problem_row.get('ground_truth_answer')
    has_gt = ground_truth not in (None, '', 'None')
    answer_is_correct = False
    if predicted_label == 'Determinate' and has_gt:
        answer_is_correct = bool(rng.random() < min(0.97, max(0.45, quality + 0.10)))
        final_answer = str(ground_truth) if answer_is_correct else f'Alternative estimate {int(local_seed % 97)}'
    elif predicted_label == 'Determinate':       final_answer = 'A single answer appears likely.'
    elif predicted_label == 'Underdetermined':   final_answer = 'The prompt supports multiple defensible answers.'
    elif predicted_label == 'Insufficient':      final_answer = 'I cannot determine a definitive answer; key information is missing.'
    else:                                        final_answer = 'The premises are internally contradictory; no coherent answer exists.'
    conf = float(rng.uniform(0.68, 0.93) if predicted_label == true_label else rng.uniform(0.35, 0.74))
    confidence_pct = int(round(max(1.0, min(99.0, conf * 100.0))))
    return (
        f'FINAL_ANSWER: {final_answer}\nSOLVABILITY_CLASS: {predicted_label}\n'
        f'CONFIDENCE: {confidence_pct}\nJUSTIFICATION_TYPE: StructuredOfflineSimulation\n'
        f'REASONING: Offline simulation; quality={quality:.2f}.'
    )


def _generate_offline_results_df(evaluation_df, model_name):
    rows = []
    run_id = f'offline-{_model_slug(model_name)}'
    for idx, row in evaluation_df.reset_index(drop=True).iterrows():
        params = dict(row)
        rows.append({'row_id': idx, **params,
                     'response': _build_offline_response(params, model_name),
                     'run_id': run_id, 'attempt_id': 0, 'attempt': 0})
    return pd.DataFrame(rows)


offline_no_model_api = bool(globals().get('USING_OFFLINE_SYNTHETIC_RESPONSES',
                                          globals().get('RUN_WITHOUT_MODELS', False)))
execution_mode = str(globals().get('EXECUTION_MODE','kaggle')).strip().lower()
if offline_no_model_api and execution_mode != 'offline_validation':
    raise RuntimeError('Synthetic responses are disabled unless EXECUTION_MODE=offline_validation.')
if offline_no_model_api:
    print('Offline mode active: model calls skipped, using synthetic responses.')

# Detect whether we will use kbench.evaluate() or direct calling
USE_KBENCH_EVALUATE = (
    globals().get('KAGGLE_KBENCH_AVAILABLE', False)
    and kbench is not None
    and not offline_no_model_api
    and execution_mode in {'kaggle'}
)
print(f'Evaluation path: {"kbench.evaluate()" if USE_KBENCH_EVALUATE else "direct API loop"}')

checkpoint_dir = str(globals().get('EPOCH1_CHECKPOINT_DIR', 'epoch1_model_checkpoints'))
resume_from_checkpoint = bool(globals().get('RESUME_FROM_CHECKPOINT', True))
os.makedirs(checkpoint_dir, exist_ok=True)

skip_models = set(str(m) for m in globals().get('SKIP_MODELS', []))
active_models = [(mn, mo) for mn, mo in models_to_run if mn not in skip_models]
if not active_models:
    raise RuntimeError('No models left to run after SKIP_MODELS filter.')

print(f'Active models: {len(active_models)}')
for mn, _ in active_models: print(' -', mn)
print(f'Time remaining: {time_remaining()/60:.0f} min')

for model_name, model_obj in active_models:
    ckpt_file = os.path.join(checkpoint_dir, f'{_model_slug(model_name)}.csv')

    if not time_ok():
        print(f'\nTime budget reached; skipping {model_name}.')
        failed_models.append({'model': model_name, 'error': 'TIME_BUDGET_EXHAUSTED'}); break

    if resume_from_checkpoint and os.path.exists(ckpt_file):
        try:
            cached_df = pd.read_csv(ckpt_file)
            if {'problem_id','response'}.issubset(set(cached_df.columns)):
                print(f'\nLoading checkpoint for {model_name} ({len(cached_df)} rows)')
                run_outputs[model_name] = {'runs_obj': None, 'results_df': cached_df, 'model_obj': model_obj}
                continue
        except Exception:
            pass

    print('\n' + '='*80); print(f'Evaluating: {model_name}'); print('='*80)
    _nudge_text = _get_format_reinforcement(model_name) if '_get_format_reinforcement' in dir() else ''
    model_df = df.copy()
    if _nudge_text: model_df['system'] = model_df['system'].astype(str) + _nudge_text
    model_start = _time.time()

    # ── Path A: Offline synthetic
    if offline_no_model_api:
        try:
            runs_obj   = None
            results_df = _generate_offline_results_df(model_df, model_name)
            print(f'  Synthesized {len(results_df)} offline rows in {(_time.time()-model_start)/60:.1f} min')
        except Exception as e:
            print(f'  Offline synthesis failed: {type(e).__name__}')
            failed_models.append({'model': model_name, 'error': f'{type(e).__name__}: {e}'}); continue

    # ── Path B: Kaggle kbench.evaluate()
    elif USE_KBENCH_EVALUATE:
        try:
            runs_obj = prometheus_ebm.evaluate(
                evaluation_data=model_df, grid={'llm': [model_obj]}, n_jobs=1, max_attempts=1
            )
        except Exception as e:
            elapsed = _time.time() - model_start
            print(f'  {model_name} failed after {elapsed/60:.1f} min: {type(e).__name__}')
            failed_models.append({'model': model_name, 'error': f'{type(e).__name__}: {e}'}); continue
        elapsed = _time.time() - model_start
        try:
            runs_list = list(getattr(runs_obj, 'runs', []))
        except Exception as e:
            failed_models.append({'model': model_name, 'error': f'RUN_COLLECTION_ERROR: {type(e).__name__}: {e}'}); continue
        print(f'  {len(runs_list)} evaluations completed in {elapsed/60:.1f} min')
        try:
            rows = []
            for idx, r in enumerate(runs_list):
                params = dict(getattr(r, 'params', {}) or {})
                rows.append({'row_id': idx, **params, 'response': run_to_response_text(r)})
            results_df = pd.DataFrame(rows)
        except Exception as e:
            failed_models.append({'model': model_name, 'error': f'RESULTS_BUILD_ERROR: {type(e).__name__}: {e}'}); continue

    # ── Path C: Direct API loop (Colab / local / non-Kaggle)
    else:
        try:
            rows = []
            total = len(model_df)
            for idx, row in model_df.iterrows():
                if (idx + 1) % 10 == 0 or idx == total - 1:
                    print(f'  [{_model_slug(model_name)[:15]}] {idx+1}/{total} '
                          f'(elapsed {(_time.time()-model_start)/60:.1f} min)')
                params = dict(row)
                result = prometheus_ebm_task(
                    model_obj,
                    problem_id             = str(params.get('problem_id', '')),
                    system                 = str(params.get('system', '')),
                    user                   = str(params.get('user', '')),
                    ground_truth_answer    = params.get('ground_truth_answer', ''),
                    correct_solvability_class = str(params.get('correct_solvability_class', '')),
                    problem_class          = str(params.get('problem_class', '')),
                    domain                 = str(params.get('domain', '')),
                    branching_factor       = int(params.get('branching_factor', 2)),
                    rigor_mode             = str(params.get('rigor_mode', 'base')),
                )
                rows.append({
                    'row_id':    idx,
                    **params,
                    'response':  result['response'],
                    'run_id':    f'direct-{_model_slug(model_name)}',
                    'attempt_id': 0,
                    'attempt':   0,
                })
            runs_obj   = None
            results_df = pd.DataFrame(rows)
            print(f'  {len(results_df)} direct evaluations complete in {(_time.time()-model_start)/60:.1f} min')
        except Exception as e:
            print(f'  Direct eval failed: {type(e).__name__}: {e}')
            failed_models.append({'model': model_name, 'error': f'{type(e).__name__}: {e}'}); continue

    if len(results_df) > 0:
        print(results_df[['problem_id','response']].head(2).to_string(index=False))
    try:
        results_df.to_csv(ckpt_file, index=False); print('  Checkpoint saved.')
    except Exception:
        print(f'  Checkpoint save failed for {model_name}')
    run_outputs[model_name] = {'runs_obj': runs_obj, 'results_df': results_df, 'model_obj': model_obj}

print('\nEvaluation complete.')
print(f'Models evaluated: {list(run_outputs.keys())}')
print(f'Total evaluation time: {(_time.time()-SESSION_START_TIME)/60:.1f} min')
if failed_models:
    print('Models with errors:')
    for f in failed_models: print(' -', f['model'], '=>', f['error'])
else:
    print('All models completed without errors.')

---
## C10 onwards — identical to Final_V5.ipynb
All scoring, visualization, export, and Epoch-2 cells below are the same as the canonical V5 notebook. The only cells that differ in this portable edition are **C03** (env setup), **C08** (task definition without @kbench.task), and **C09** (platform-adaptive eval loop).

In [ ]:
# [C10] Offline ECI and HGI Scoring
from collections import defaultdict

def compute_hysteresis_gap(results_df, prompts_df):
    prompts_local = prompts_df.reset_index(drop=True).copy()
    out = prompts_local[['problem_id','problem_class','domain']].copy()
    if 'response' not in results_df.columns:
        raise ValueError(f'No response column. Available: {results_df.columns.tolist()}')
    raw = results_df['response'].astype(str).tolist()
    if len(raw) != len(out):
        raise ValueError(f'Length mismatch: responses={len(raw)} prompts={len(out)}')
    parsed_rows = []
    for i, r in enumerate(raw):
        p = parse_response(out.loc[i,'problem_id'], r)
        is_correct, _ = evaluate_answer_correctness(
            p.final_answer, prompts_local.loc[i,'ground_truth_answer'],
            prompts_local.loc[i,'problem_class'], p.solvability_class_estimate, judge_fn=None)
        solv_ok    = (p.solvability_class_estimate == prompts_local.loc[i,'correct_solvability_class'])
        conf       = p.confidence if p.confidence is not None else 0.5
        conf_aligned = 1.0 - abs(conf - (1.0 if is_correct else 0.0))
        gap = (abs((1.0 if is_correct else 0.0) - conf_aligned) +
               abs((1.0 if is_correct else 0.0) - (1.0 if solv_ok else 0.0))) / 2.0
        parsed_rows.append({'correct': float(1.0 if is_correct else 0.0), 'confidence': float(conf),
                            'confidence_alignment': float(conf_aligned),
                            'solvability_correct': float(1.0 if solv_ok else 0.0), 'hysteresis_gap': float(gap)})
    out = pd.concat([out, pd.DataFrame(parsed_rows)], axis=1)
    by_domain = out.groupby('domain',as_index=False)['hysteresis_gap'].mean().rename(columns={'hysteresis_gap':'mean_hysteresis_gap'})
    by_class  = out.groupby('problem_class',as_index=False)['hysteresis_gap'].mean().rename(columns={'hysteresis_gap':'mean_hysteresis_gap'})
    return {'per_item': out, 'by_domain': by_domain, 'by_problem_class': by_class, 'HGI': float(out['hysteresis_gap'].mean())}

summary_rows = []; details = {}
for model_name, payload in run_outputs.items():
    results_df = payload['results_df']
    if 'problem_id' not in results_df.columns or 'response' not in results_df.columns:
        print(f'Skipping {model_name}: missing columns'); continue
    resp_map        = dict(zip(results_df['problem_id'], results_df['response']))
    aligned_responses = [resp_map.get(p['problem_id'],'') for p in prompts]
    scorer          = ECIScorer(model_name=model_name, epoch=EPOCH)
    eci             = scorer.score(prompts, aligned_responses, judge_fn=None)
    aligned_df      = pd.DataFrame({'problem_id':[p['problem_id'] for p in prompts], 'response': aligned_responses})
    hgi_obj         = compute_hysteresis_gap(aligned_df, df)
    summary_rows.append({'model': model_name, 'n': len(aligned_responses),
                         'eci': float(eci.eci_score), 'sda': float(eci.solvability_detection_accuracy),
                         'ca': float(eci.conditional_accuracy), 'rp': float(eci.refusal_precision),
                         'ece': float(eci.expected_calibration_error),
                         'hss': float(eci.hallucination_severity_score), 'hgi': float(hgi_obj['HGI'])})
    details[model_name] = {'eci': eci, 'hgi': hgi_obj}

if len(summary_rows) == 0:
    summary_df = pd.DataFrame(columns=['model','n','eci','sda','ca','rp','ece','hss','hgi'])
    print('No successful model summaries produced.')
else:
    summary_df = pd.DataFrame(summary_rows).sort_values('eci', ascending=False).reset_index(drop=True)
    print('\nModel Comparison Summary'); print(summary_df.to_string(index=False))

In [ ]:
# [C11] Visualizations and CSV Export
if len(summary_df) == 0:
    raise RuntimeError('No model summaries produced.')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(summary_df['model'], summary_df['eci'])
axes[0].set_title('ECI by Model'); axes[0].set_ylim(0,1); axes[0].tick_params(axis='x', rotation=20)
axes[1].bar(summary_df['model'], summary_df['hgi'])
axes[1].set_title('HGI by Model (lower is better)'); axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()
summary_df.to_csv('prometheus_model_comparison.csv', index=False)
print('Saved: prometheus_model_comparison.csv')

item_rows = []
for model_name, payload in run_outputs.items():
    results_df = payload.get('results_df', pd.DataFrame()).copy()
    if len(results_df) == 0 or 'problem_id' not in results_df.columns: continue
    first_by_problem = {}
    for _, r in results_df.iterrows():
        pid = r.get('problem_id')
        if pid not in first_by_problem: first_by_problem[pid] = r
    for p in prompts:
        pid = p.get('problem_id')
        row = first_by_problem.get(pid, pd.Series(dtype='object'))
        raw_response = str(row.get('response',''))
        parsed = parse_response(pid, raw_response)
        gt = p.get('ground_truth_answer'); gt_norm = None if gt in (None,'','None') else gt
        is_correct, _ = evaluate_answer_correctness(
            parsed.final_answer, gt_norm, p.get('problem_class',''), parsed.solvability_class_estimate, judge_fn=None)
        item_rows.append({'model': model_name, 'problem_id': pid,
                          'final_answer': parsed.final_answer, 'ground_truth': gt_norm,
                          'correctness_flag': bool(is_correct), 'solvability_class': parsed.solvability_class_estimate,
                          'confidence': parsed.confidence, 'justification_type': parsed.justification_type,
                          'reasoning_text': parsed.reasoning, 'run_id': row.get('run_id'),
                          'attempt_id': row.get('attempt_id', row.get('attempt', row.get('row_id'))),
                          'problem_class': p.get('problem_class',''), 'domain': p.get('domain',''),
                          'rigor_mode': p.get('rigor_mode','base')})
item_level_df = pd.DataFrame(item_rows)
item_level_df.to_csv('prometheus_item_level_results.csv', index=False)
print('Saved: prometheus_item_level_results.csv')
print('Item-level rows:', len(item_level_df))

brier_df = pd.DataFrame()
if len(item_level_df) > 0:
    brier_rows = []
    for model_name in item_level_df['model'].unique():
        mdf = item_level_df[item_level_df['model'] == model_name].copy()
        mdf['confidence']      = pd.to_numeric(mdf['confidence'], errors='coerce')
        mdf['correctness_flag'] = pd.to_numeric(mdf['correctness_flag'], errors='coerce').fillna(0).astype(int)
        mdf = mdf.dropna(subset=['confidence'])
        confs = mdf['confidence'].astype(float).tolist()
        flags = mdf['correctness_flag'].astype(int).tolist()
        if confs:
            bd = brier_score_decomposition(confs, flags)
            dp = type2_d_prime(confs, flags)
            brier_rows.append({'model': model_name, 'brier_score': bd['brier'],
                               'brier_reliability': bd['reliability'], 'brier_resolution': bd['resolution'],
                               'brier_uncertainty': bd['uncertainty'], 'd_prime': dp['d_prime'],
                               'hit_rate': dp['hit_rate'], 'false_alarm_rate': dp['false_alarm_rate']})
    if brier_rows:
        brier_df = pd.DataFrame(brier_rows).sort_values('d_prime', ascending=False)
        brier_df.to_csv('prometheus_brier_dprime.csv', index=False)
        print('Saved: prometheus_brier_dprime.csv')
        summary_df = summary_df.merge(brier_df[['model','brier_score','d_prime']], on='model', how='left')
        summary_df.to_csv('prometheus_model_comparison.csv', index=False)
        print('Updated: prometheus_model_comparison.csv (with brier_score and d_prime)')

if len(item_level_df) > 0:
    def _col_as_series(df_obj, col):
        return pd.to_numeric(df_obj[col], errors='coerce') if col in df_obj.columns else pd.Series(np.nan, index=df_obj.index, dtype=float)
    eci_series   = _col_as_series(summary_df,'eci').fillna(0.0)
    probe_series = _col_as_series(summary_df,'probe_accuracy')
    if probe_series.isna().all(): probe_series = _col_as_series(summary_df,'ca')
    probe_series = probe_series.fillna(0.0)
    rp_series  = _col_as_series(summary_df,'rp').fillna(0.0)
    ece_series = _col_as_series(summary_df,'ece').fillna(0.0)
    hgi_series = _col_as_series(summary_df,'hgi').fillna(0.0)
    hgi_min = float(hgi_series.min()); hgi_max = float(hgi_series.max())
    if abs(hgi_max - hgi_min) > 1e-9:
        hgi_norm = (hgi_series - hgi_min) / (hgi_max - hgi_min)
    else: hgi_norm = pd.Series(np.zeros(len(summary_df)), index=summary_df.index)
    readiness_raw = (0.35*eci_series + 0.20*probe_series + 0.15*(1.0-hgi_norm) + 0.15*(1.0-ece_series) + 0.15*rp_series)
    summary_df['metacog_readiness_score'] = readiness_raw.clip(0.0, 1.0)
    def _readiness_tier(v):
        if pd.isna(v): return 'unrated'
        if v >= 0.75:  return 'frontier_metacognitive_reliability'
        if v >= 0.60:  return 'strong_metacognitive_reliability'
        return 'exploratory'
    summary_df['metacog_readiness_tier'] = summary_df['metacog_readiness_score'].map(_readiness_tier)
    summary_df.to_csv('prometheus_model_comparison.csv', index=False)
    print('Updated: prometheus_model_comparison.csv (with readiness scores)')

if len(summary_df) > 0:
    top_model = summary_df.iloc[0]['model']
    print('\nTop model:', top_model)
    print(details[top_model]['eci'].summary())

In [ ]:
# [C11.5] Epistemic Fingerprint Radar Chart
if len(summary_df) == 0: raise RuntimeError('No model summaries produced.')
radar_df = summary_df.copy()
if 'overconfidence_gap' not in radar_df.columns or radar_df['overconfidence_gap'].isna().all():
    if 'item_level_df' in globals() and len(item_level_df) > 0:
        gap_df = (item_level_df.groupby('model',as_index=False)
                  .agg(mean_correctness=('correctness_flag', lambda s: float(pd.to_numeric(s,errors='coerce').mean())),
                       mean_confidence=('confidence', lambda s: float(pd.to_numeric(s,errors='coerce').mean()))))
        gap_df['overconfidence_gap'] = gap_df['mean_confidence'] - gap_df['mean_correctness']
        radar_df = radar_df.merge(gap_df[['model','overconfidence_gap']], on='model', how='left')
    else: radar_df['overconfidence_gap'] = np.nan
radar_df['overconfidence_gap'] = pd.to_numeric(radar_df['overconfidence_gap'], errors='coerce').fillna(0.0)
metrics = ['sda','ca','rp','overconfidence_gap']
labels  = ['SDA','CA','RP','Calibration (1 - gap)']
angles  = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist(); angles += angles[:1]
fig = plt.figure(figsize=(8,8)); ax = plt.subplot(111, polar=True)
for _, row in radar_df.iterrows():
    values = []
    for metric in metrics:
        val = pd.to_numeric(row.get(metric), errors='coerce')
        if metric == 'overconfidence_gap': values.append(float(np.clip(1.0 - float(val) if pd.notna(val) else 1.0, 0, 1)))
        else: values.append(float(val) if pd.notna(val) else 0.0)
    values += values[:1]
    model_label = str(row['model']).split('/')[-1]
    ax.plot(angles, values, linewidth=2, label=model_label); ax.fill(angles, values, alpha=0.08)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels); ax.set_ylim(0,1)
ax.set_title('Epistemic Fingerprint Radar\n(Higher is better on all axes)', pad=24)
ax.legend(loc='upper right', bbox_to_anchor=(1.3,1.1)); plt.tight_layout(); plt.show()

In [ ]:
# [C12] Output Contract Validator
import os, pandas as pd

final_output_basename = str(globals().get('FINAL_OUTPUT_BASENAME','Final_Output_main')).strip() or 'Final_Output_main'
final_output_csv  = f'{final_output_basename}.csv'
final_output_json = f'{final_output_basename}.json'

PRE_EXPORT_REQUIRED = ['prometheus_model_comparison.csv','prometheus_item_level_results.csv','prometheus_brier_dprime.csv']

def validate_outputs(required_files):
    missing = [f for f in required_files if not os.path.exists(f)]
    return {'ok': len(missing)==0, 'missing': missing, 'present': [f for f in required_files if f not in missing]}

def validate_csv_schema(item_csv='prometheus_item_level_results.csv', summary_csv='prometheus_model_comparison.csv'):
    schema = {'ok': True, 'problems': []}
    if os.path.exists(item_csv):
        item = pd.read_csv(item_csv)
        req  = ['model','problem_id','final_answer','ground_truth','correctness_flag',
                'solvability_class','confidence','justification_type','reasoning_text','run_id','attempt_id']
        miss = [c for c in req if c not in item.columns]
        if miss: schema['ok'] = False; schema['problems'].append(f'item-level missing: {miss}')
    else: schema['ok'] = False; schema['problems'].append('item-level csv missing')
    if os.path.exists(summary_csv):
        summary = pd.read_csv(summary_csv)
        req     = ['model','eci','sda','ca','rp','ece','hss','hgi']
        miss    = [c for c in req if c not in summary.columns]
        if miss: schema['ok'] = False; schema['problems'].append(f'summary missing: {miss}')
    else: schema['ok'] = False; schema['problems'].append('summary csv missing')
    return schema

pre_export_check = validate_outputs(PRE_EXPORT_REQUIRED)
schema_check     = validate_csv_schema()
SCHEMA_VALIDATION_OK = bool(pre_export_check['ok'] and schema_check['ok'])
print('Pre-export check:', pre_export_check)
print('Schema check:', schema_check)
print('SCHEMA_VALIDATION_OK:', SCHEMA_VALIDATION_OK)

In [ ]:
# [C13] Multi-Stage Evaluation Scaffold (A/B/C/D)
import re

def _extract_confidence_from_response(raw_text, default=0.5):
    text = str(raw_text or '')
    m    = re.search(r'CONFIDENCE\s*:\s*([0-9]+(?:\.[0-9]+)?)', text, flags=re.IGNORECASE)
    if not m: return float(default)
    val = float(m.group(1))
    return float(max(0.0, min(1.0, val/100.0 if val > 1.0 else val)))

def _safe_prompt_text(llm, user_text, system_text=None):
    if 'safe_prompt' in globals():
        return str(safe_prompt(llm, user_text=user_text, system_text=system_text, retries=2))
    if system_text:
        try: return str(llm.prompt(user_text, system=system_text))
        except TypeError: return str(llm.prompt(system_text+'\n\n'+user_text))
    return str(llm.prompt(user_text))

print('Multi-stage adversarial protocol initialized.')

In [ ]:
# [C14] Statistical Utilities: Bootstrap Confidence Intervals
import numpy as np, pandas as pd

def bootstrap_ci(values, n_boot=1000, alpha=0.05, seed=42):
    arr = np.array([float(v) for v in values if pd.notna(v)], dtype=float)
    if len(arr) == 0: return {'mean':np.nan,'ci_low':np.nan,'ci_high':np.nan,'std':np.nan,'n':0}
    rng = np.random.default_rng(seed)
    boot_means = [float(np.mean(rng.choice(arr, size=len(arr), replace=True))) for _ in range(int(n_boot))]
    return {'mean':float(np.mean(arr)),'ci_low':float(np.quantile(boot_means,alpha/2.0)),
            'ci_high':float(np.quantile(boot_means,1.0-alpha/2.0)),'std':float(np.std(arr)),'n':int(len(arr))}

print('Bootstrap CI utilities initialized.')

In [ ]:
# [C15] Instrumentation: Logging, Run Manifest
import os, json, time
from datetime import datetime, timezone

RUN_LOG_DIR       = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
RUN_MANIFEST_PATH = os.path.join(RUN_LOG_DIR, 'prometheus_run_manifest.jsonl')

def append_run_event(event_type, payload):
    event = {'ts_utc': datetime.now(timezone.utc).isoformat(), 'event_type': event_type, 'payload': payload}
    with open(RUN_MANIFEST_PATH, 'a', encoding='utf-8') as f:
        f.write(json.dumps(event, ensure_ascii=False) + '\n')

print(f'Run manifest: {RUN_MANIFEST_PATH}')

In [ ]:
# [C16] Calibration and Failure Analysis
import numpy as np, pandas as pd, matplotlib.pyplot as plt

def plot_calibration_curve(item_df, correctness_col='correctness_flag', confidence_col='confidence', n_bins=10):
    d = item_df[[correctness_col,confidence_col]].dropna().copy()
    if len(d) == 0: print('No data.'); return
    d[correctness_col] = d[correctness_col].astype(float)
    d[confidence_col]  = d[confidence_col].astype(float).clip(0.0,1.0)
    d['bin'] = pd.cut(d[confidence_col], bins=np.linspace(0,1,n_bins+1), include_lowest=True)
    g = d.groupby('bin',observed=False).agg(
        empirical_accuracy=(correctness_col,'mean'), mean_confidence=(confidence_col,'mean'),
        count=(correctness_col,'size')).reset_index()
    plt.figure(figsize=(6,6))
    plt.plot([0,1],[0,1],'--',label='Perfect calibration')
    plt.plot(g['mean_confidence'],g['empirical_accuracy'],marker='o',label='Observed')
    plt.xlabel('Mean confidence'); plt.ylabel('Empirical accuracy')
    plt.title('Calibration Curve'); plt.legend(); plt.grid(alpha=0.25); plt.show()

def plot_failure_heatmap(item_df):
    d = item_df.copy()
    if 'correctness_flag' not in d.columns: print('Missing correctness_flag.'); return
    d['error'] = 1.0 - d['correctness_flag'].astype(float)
    pivot = d.pivot_table(index='problem_class', columns='model', values='error', aggfunc='mean')
    if pivot.empty: print('No data.'); return
    plt.figure(figsize=(10,4))
    plt.imshow(pivot.values, aspect='auto')
    plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=20, ha='right')
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.colorbar(label='Mean failure rate')
    plt.title('Failure Heatmap (problem_class x model)'); plt.tight_layout(); plt.show()

print('Calibration and failure analysis initialized.')

In [ ]:
# [C17] Artifact Export Bundle + Unified Final Output
import os, json, zipfile, numpy as np, pandas as pd
from datetime import datetime, timezone

required = ['prometheus_item_level_results.csv','prometheus_model_comparison.csv']
missing  = [f for f in required if not os.path.exists(f)]
if missing: raise FileNotFoundError(f'Missing required files: {missing}')

item_df    = pd.read_csv('prometheus_item_level_results.csv')
summary_df = pd.read_csv('prometheus_model_comparison.csv')
item_df.to_json('prometheus_item_level_results.json', orient='records', indent=2)
summary_df.to_json('prometheus_model_comparison.json', orient='records', indent=2)

generated_at      = datetime.now(timezone.utc).isoformat()
benchmark_mode    = str(globals().get('BENCHMARK_MODE',''))
run_scope         = str(globals().get('RUN_SCOPE',''))
model_provider    = str(globals().get('MODEL_PROVIDER','unknown'))
pairwise_required = bool(globals().get('PAIRWISE_REQUIRED', len(summary_df) > 1))
target_models     = list(globals().get('TARGET_MODELS',[]))
agi_target        = float(np.clip(globals().get('AGI_METACOG_TARGET_SCORE', 0.85), 0.0, 1.0))
final_output_basename = str(globals().get('FINAL_OUTPUT_BASENAME','Final_Output_main')).strip() or 'Final_Output_main'
final_output_csv  = f'{final_output_basename}.csv'
final_output_json = f'{final_output_basename}.json'

final_df = summary_df.copy()
for col, default in {'eci':np.nan,'sda':np.nan,'ca':np.nan,'rp':np.nan,'ece':np.nan,'hss':np.nan,
                     'hgi':np.nan,'brier_score':np.nan,'d_prime':np.nan,
                     'metacog_readiness_score':np.nan,'metacog_readiness_tier':'unrated'}.items():
    if col not in final_df.columns: final_df[col] = default

for col in ['eci','sda','ca','rp','ece','hss','hgi','brier_score','d_prime']:
    final_df[col] = pd.to_numeric(final_df[col], errors='coerce')

readiness = pd.to_numeric(final_df['metacog_readiness_score'], errors='coerce')
readiness = readiness.fillna(pd.to_numeric(final_df['eci'], errors='coerce').fillna(0.0))
final_df['metacog_readiness_score'] = readiness.clip(0.0,1.0)

final_df = final_df.sort_values('eci', ascending=False).reset_index(drop=True)
final_df['run_rank_by_eci'] = np.arange(1, len(final_df)+1, dtype=int)
if len(final_df) > 0 and final_df['eci'].notna().any():
    eci_leader = float(final_df['eci'].max())
    final_df['eci_gap_to_run_leader'] = (eci_leader - final_df['eci']).clip(lower=0.0)
else: final_df['eci_gap_to_run_leader'] = np.nan

final_df['agi_target_score'] = agi_target
final_df['agi_score_gap']    = agi_target - final_df['metacog_readiness_score']
if agi_target > 0:
    final_df['agi_progress_pct'] = (100.0 * final_df['metacog_readiness_score'] / agi_target).clip(lower=0.0)
else: final_df['agi_progress_pct'] = np.nan
final_df['agi_target_met']      = final_df['metacog_readiness_score'] >= agi_target
final_df['generated_at_utc']    = generated_at
final_df['benchmark_mode']      = benchmark_mode
final_df['run_scope']           = run_scope
final_df['model_provider']      = model_provider
final_df['pairwise_required']   = pairwise_required
final_df['target_model_count']  = len(target_models)
final_df['target_models']       = '|'.join([str(x) for x in target_models])
final_df['comparison_mode']     = 'multi_model' if len(final_df) > 1 else 'solo_model'

final_df.to_csv(final_output_csv, index=False)
final_df.to_json(final_output_json, orient='records', indent=2)

run_meta = {'generated_at_utc': generated_at, 'item_rows': int(len(item_df)),
            'summary_rows': int(len(summary_df)), 'final_output_rows': int(len(final_df)),
            'benchmark_mode': benchmark_mode, 'run_scope': run_scope,
            'model_provider': model_provider, 'target_model_count': len(target_models),
            'models': sorted(summary_df['model'].astype(str).tolist()) if 'model' in summary_df.columns else [],
            'agi_metacog_target_score': agi_target,
            'final_output_csv': final_output_csv, 'final_output_json': final_output_json}
with open('prometheus_export_manifest.json','w',encoding='utf-8') as f:
    json.dump(run_meta, f, indent=2)

run_profile = {**{k: run_meta[k] for k in ['generated_at_utc','benchmark_mode','run_scope',
                                            'model_provider','target_model_count','models',
                                            'agi_metacog_target_score']},
               'pairwise_required': pairwise_required,
               'offline_no_model_mode': bool(globals().get('USING_OFFLINE_SYNTHETIC_RESPONSES', False))}
with open('run_profile.json','w',encoding='utf-8') as f:
    json.dump(run_profile, f, indent=2)

bundle_files = ['prometheus_item_level_results.csv','prometheus_model_comparison.csv',
                'prometheus_item_level_results.json','prometheus_model_comparison.json',
                final_output_csv, final_output_json, 'prometheus_export_manifest.json','run_profile.json']
zip_name = 'prometheus_results_export.zip'
with zipfile.ZipFile(zip_name, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for fn in bundle_files:
        if os.path.exists(fn): zf.write(fn)
print('Saved:', zip_name, '|', final_output_csv, '|', final_output_json)
print('Export complete.')

# Epoch-2: Targeted Epistemic Stress Probes

Epoch-1 measures overall epistemic calibration on a broad item set. Epoch-2 asks a sharper question: **do those findings hold under deliberate epistemic stress?**

| Track | What It Probes | Failure Signature |
|---|---|---|
| **Ambiguity Probe** | UNDERDETERMINED — multiple valid answers | Forcing a singular answer = overconfidence |
| **Contradiction Probe** | CONTRADICTORY — conflicting premises | Answering without flagging conflict = hallucination |
| **Multi-Stage Protocol** | Belief revision under adversarial pressure | Flip under pressure = fragility; refuse valid evidence = dogmatism |

All Epoch-2 artifacts are generated independently and never merged into Epoch-1 outputs.

In [ ]:
# [P01] Load Probe Datasets
import json, os, glob

def find_probe_file(name):
    search_paths = [
        name, os.path.join(os.getcwd(), name),
        os.path.join('/kaggle/input', name), os.path.join('/content', name),
        f'/kaggle/input/**/{name}', f'/content/**/{name}',
    ]
    for pattern in search_paths:
        if '*' in pattern:
            matches = glob.glob(pattern, recursive=True)
            if matches: return matches[0]
        elif os.path.isfile(pattern):
            return pattern
    return None

ambiguity_path    = find_probe_file('probe_ambiguity.json')
contradiction_path = find_probe_file('probe_contradictions.json')
probe_problems = []
for path, label in [(ambiguity_path,'ambiguity'), (contradiction_path,'contradiction')]:
    if path:
        with open(path,'r') as f: data = json.load(f)
        print(f'Loaded {len(data)} {label} probe problems from {path}')
        probe_problems.extend(data)
    else:
        print(f'WARNING: probe_{label}.json not found')
print(f'\nTotal probe problems: {len(probe_problems)}')

In [ ]:
# [P02] Build Probe Prompts
probe_prompts = []
for p in probe_problems:
    probe_prompts.append({
        'problem_id':               p['problem_id'],
        'system':                   p.get('system',''),
        'user':                     p.get('user', p.get('question','')),
        'ground_truth_answer':      p.get('ground_truth_answer'),
        'correct_solvability_class': p.get('correct_solvability_class', p.get('problem_class')),
        'problem_class':            p.get('problem_class'),
        'domain':                   p.get('domain'),
        'branching_factor':         p.get('branching_factor', 2),
        'rigor_mode':               'base',
    })
print(f'Built {len(probe_prompts)} probe prompts')
print(f'  UNDERDETERMINED: {sum(1 for p in probe_prompts if p["problem_class"]=="UNDERDETERMINED")}')
print(f'  CONTRADICTORY:   {sum(1 for p in probe_prompts if p["problem_class"]=="CONTRADICTORY")}')

In [ ]:
# [P02.5] Epoch-2 Compatibility Layer
import re

if 'SYSTEM_PROMPT' not in globals():
    SYSTEM_PROMPT = ('You are solving PROMETHEUS-EBM tasks. Always answer using exact fields:\n'
                     'FINAL_ANSWER, SOLVABILITY_CLASS, CONFIDENCE, JUSTIFICATION_TYPE, REASONING.')

_probe_llm_cache = {}

def _normalize_confidence(value, default=0.5):
    try: v = float(value)
    except Exception: return float(default)
    if v > 1.0: v = v / 100.0
    return float(max(0.0, min(1.0, v)))

def _normalize_solvability_label(value):
    s = str(value or '').strip().lower()
    if not s: return None
    if 'under' in s: return 'Underdetermined'
    if 'insuff' in s or 'not enough' in s: return 'Insufficient'
    if 'contrad' in s or 'inconsist' in s: return 'Contradictory'
    if 'determin' in s: return 'Determinate'
    return None

def _infer_solvability_from_text(*parts):
    text = ' '.join(str(p or '') for p in parts).lower()
    if not text.strip(): return None
    if any(t in text for t in ['contradict','inconsistent','conflict']): return 'Contradictory'
    if any(t in text for t in ['insufficient','not enough','cannot be determined','unanswerable']): return 'Insufficient'
    if any(t in text for t in ['underdetermined','multiple plausible','ambiguous']): return 'Underdetermined'
    if any(t in text for t in ['determinate','the answer is']): return 'Determinate'
    return None

def _resolve_llm_from_model_id(model_id):
    if model_id in _probe_llm_cache: return _probe_llm_cache[model_id]
    pool = []
    if 'all_pool' in globals() and isinstance(all_pool, list): pool = list(all_pool)
    model_lower = str(model_id).lower()
    exact = [(n,o) for (n,o) in pool if str(n).lower() == model_lower]
    if exact: _probe_llm_cache[model_id] = exact[0][1]; return exact[0][1]
    partial = [(n,o) for (n,o) in pool if model_lower in str(n).lower() or str(n).lower() in model_lower]
    if partial: chosen = sorted(partial, key=lambda x: len(str(x[0])))[0][1]; _probe_llm_cache[model_id] = chosen; return chosen
    return None

def call_openrouter(model_id, messages):
    llm = _resolve_llm_from_model_id(model_id)
    if llm is None:
        return ('FINAL_ANSWER: REFUSAL\nSOLVABILITY_CLASS: Insufficient\nCONFIDENCE: 5\n'
                'JUSTIFICATION_TYPE: Refusal\nREASONING: Model not resolved.')
    system_parts = [str(m.get('content','')) for m in messages if m.get('role')=='system']
    system_text  = '\n\n'.join([s for s in system_parts if s]).strip()
    history = []
    for m in messages:
        if m.get('role') == 'system': continue
        history.append(f'== {str(m.get("role","user")).upper()} ==\n{str(m.get("content","")).strip()}')
    user_text = '\n\n'.join(history).strip()
    if 'safe_prompt' in globals():
        try: return str(safe_prompt(llm, user_text=user_text, system_text=system_text, retries=2))
        except Exception as e: return f'FINAL_ANSWER: EVALUATION_ERROR\nSOLVABILITY_CLASS: Evaluation_Error\nCONFIDENCE: 0\nJUSTIFICATION_TYPE: Error\nREASONING: {type(e).__name__}'
    try: return str(llm.prompt(user_text, system=system_text))
    except Exception as e: return f'FINAL_ANSWER: EVALUATION_ERROR\nSOLVABILITY_CLASS: Evaluation_Error\nCONFIDENCE: 0\nJUSTIFICATION_TYPE: Error\nREASONING: {type(e).__name__}'

def parse_structured_response(raw_text):
    raw   = str(raw_text or '')
    final_answer = solvability = just = reason = None
    conf  = 0.5; parse_route = 'none'
    if 'parse_response' in globals():
        try:
            parsed = parse_response('PROBE_RUNTIME', raw)
            final_answer = getattr(parsed,'final_answer',None)
            solvability  = _normalize_solvability_label(getattr(parsed,'solvability_class_estimate',None))
            conf         = _normalize_confidence(getattr(parsed,'confidence',0.5))
            just         = getattr(parsed,'justification_type',None)
            reason       = getattr(parsed,'reasoning',None)
            parse_route  = 'primary_parser'
        except Exception: parse_route = 'primary_parser_error'
    def _extract(field):
        m = re.search(rf'{field}:\s*(.+?)(?=\n[A-Z_]+:|$)', raw, flags=re.IGNORECASE|re.DOTALL)
        return m.group(1).strip() if m else None
    if final_answer is None: final_answer = _extract('FINAL_ANSWER')
    if just is None: just = _extract('JUSTIFICATION_TYPE')
    if reason is None: reason = _extract('REASONING')
    solv_raw = _extract('SOLVABILITY_CLASS')
    if solvability is None and solv_raw: solvability = _normalize_solvability_label(solv_raw); parse_route = 'regex_solvability'
    conf_raw = _extract('CONFIDENCE')
    if conf_raw:
        nums = re.findall(r'\d+\.?\d*', conf_raw)
        if nums: conf = _normalize_confidence(nums[0], default=conf)
    if solvability is None:
        inferred = _infer_solvability_from_text(raw, final_answer, reason, just)
        if inferred: solvability = inferred; parse_route = 'heuristic_recovery'
    if final_answer is None and raw.strip():
        candidates = [ln.strip() for ln in raw.splitlines() if ln.strip()]
        non_schema = [ln for ln in candidates if ':' not in ln[:40]]
        if non_schema: final_answer = non_schema[0][:400]; parse_route = 'answer_salvage'
    return {'final_answer': final_answer, 'solvability_class_estimate': solvability, 'confidence': conf,
            'justification_type': just, 'reasoning': reason,
            'parse_success': bool(final_answer is not None and solvability is not None),
            'schema_field_count': sum(x is not None and str(x).strip()!='' for x in [final_answer,solvability,conf_raw,just,reason]),
            'parse_route': parse_route}

print('Epoch-2 parser initialized.')

In [ ]:
# [P03] Run Probes on All Models (seeded passes)
# PROBE_MODELS = TARGET_MODELS (same models as Epoch-1 — V5 architecture)
PROBE_MODELS          = TARGET_MODELS
RUNTIME_PROBE_SEEDS   = list(globals().get('PROBE_SEEDS', [f'{SEED}-p1']))

if 'probe_prompts' not in globals() or len(probe_prompts) == 0:
    raise RuntimeError('Probe prompts are empty. Run P01 and P02 first.')

def _normalize_eval_outcome(out):
    if isinstance(out, tuple): return bool(out[0]), (out[1] if len(out)>1 else 'unknown')
    return bool(out), 'unknown'

print(f'Probe seeds: {RUNTIME_PROBE_SEEDS}')
probe_all_results = {}
for seed_value in RUNTIME_PROBE_SEEDS:
    print(f'\n--- PROBE SEED: {seed_value} ---')
    for model_id in PROBE_MODELS:
        key = f'{model_id}::{seed_value}'
        print(f'\nPROBE RUN: {model_id} | seed={seed_value} | problems={len(probe_prompts)}')
        results = []
        for i, prompt in enumerate(probe_prompts):
            if (i+1) % 10 == 0: print(f'  [{model_id.split("/")[-1][:15]}] {i+1}/{len(probe_prompts)}')
            _probe_nudge = _get_format_reinforcement(model_id) if '_get_format_reinforcement' in dir() else ''
            messages = [
                {'role':'system', 'content': SYSTEM_PROMPT+'\n\n'+prompt.get('system','')+_probe_nudge+f'\n\n[PROBE_SEED={seed_value}]'},
                {'role':'user',   'content': prompt['user']},
            ]
            raw    = call_openrouter(model_id, messages)
            parsed = parse_structured_response(raw)
            outcome = evaluate_answer_correctness(
                parsed.get('final_answer'), prompt['ground_truth_answer'],
                prompt['problem_class'], parsed.get('solvability_class_estimate'))
            is_correct, method = _normalize_eval_outcome(outcome)
            results.append({'probe_seed':seed_value,'problem_id':prompt['problem_id'],'model':model_id,
                            'problem_class':prompt['problem_class'],'domain':prompt['domain'],
                            'ground_truth':prompt['ground_truth_answer'],
                            'final_answer':parsed.get('final_answer',''),
                            'solvability_class':parsed.get('solvability_class_estimate',''),
                            'confidence':parsed.get('confidence',0.5),'correctness_flag':int(is_correct),
                            'evaluation_method':method,'parse_success':bool(parsed.get('parse_success',False)),
                            'schema_field_count':int(parsed.get('schema_field_count',0) or 0),
                            'parse_route':parsed.get('parse_route','none'),
                            'justification_type':parsed.get('justification_type',''),
                            'reasoning_text':parsed.get('reasoning',''),'rigor_mode':'probe'})
        probe_all_results[key] = results
        if results:
            correct  = sum(1 for r in results if int(r.get('correctness_flag',0))==1)
            parse_ok = sum(1 for r in results if r.get('parse_success',False))
            print(f'  Accuracy: {correct}/{len(results)} ({correct/len(results):.1%})')
            print(f'  Parse success: {parse_ok}/{len(results)} ({parse_ok/len(results):.1%})')
            for r in results:
                r['collapsed_seed'] = (len(results)>0 and (correct/len(results))<0.05 and (parse_ok/len(results))>0.90)
print('Probe runs complete.')

In [ ]:
# [P04] Score and Compare Probe Results
import json, pandas as pd

all_probe_rows = []
for model_id, results in probe_all_results.items():
    all_probe_rows.extend(results)
probe_df = pd.DataFrame(all_probe_rows)
if len(probe_df) == 0: raise RuntimeError('No probe rows generated in P03.')
if 'probe_seed' not in probe_df.columns: probe_df['probe_seed'] = globals().get('SEED','seed-unknown')

probe_df['correctness_flag']   = pd.to_numeric(probe_df['correctness_flag'],errors='coerce').fillna(0).astype(int)
probe_df['confidence']         = pd.to_numeric(probe_df['confidence'],errors='coerce').fillna(0.5).clip(0.0,1.0)
probe_df['parse_success']      = probe_df.get('parse_success',pd.Series(False,index=probe_df.index)).fillna(False).astype(bool)
probe_df['solvability_present'] = probe_df['solvability_class'].astype(str).str.strip().ne('')
probe_df['heuristic_recovered'] = probe_df.get('parse_route',pd.Series('none',index=probe_df.index)).astype(str).eq('heuristic_recovery')

clean_probe_df = probe_df[~probe_df.get('collapsed_seed',pd.Series(False,index=probe_df.index)).fillna(False)]

print('='*70)
print('EPOCH-2 PROBE RESULTS')
print('='*70)
for model_id in PROBE_MODELS:
    model_df = clean_probe_df[clean_probe_df['model'] == model_id]
    short    = model_id.split('/')[-1][:25]
    print(f'\n{short}:')
    print(f'  Overall accuracy:    {model_df["correctness_flag"].mean():.1%} (n={len(model_df)})')
    for cls in ['UNDERDETERMINED','CONTRADICTORY']:
        cls_df = model_df[model_df['problem_class']==cls]
        if len(cls_df) > 0:
            print(f'  {cls:<20}: {float(cls_df["correctness_flag"].mean()):.1%} (n={len(cls_df)}, conf={float(cls_df["confidence"].mean()):.2f})')

probe_df.to_csv('probe_results.csv', index=False)
print(f'\nSaved: probe_results.csv ({len(probe_df)} rows)')

## Multi-Stage Metacognitive Protocol

Three turns per problem:
- **Turn 1**: Standard initial response (baseline accuracy + confidence)
- **Turn 2**: Self-evaluation of previous response
- **Turn 3**: Adversarial domain-expert counter-argument

Key metrics: improvement rate, degradation rate, confidence shift, belief rigidity.

In [ ]:
# [M01] Multi-Stage Protocol (stratified sample)
import random

seed_for_multistage = f'{SEED}-multistage'
rng = random.Random(stable_int_seed(seed_for_multistage))
combined_pool = list(probe_prompts) + list(prompts[:300]) if RUN_SCOPE=='solo' else list(probe_prompts) + list(prompts[:150])
if not combined_pool: raise RuntimeError('No problems available for multi-stage protocol.')
sample_n = min(int(globals().get('MULTISTAGE_SAMPLE_N',20)), len(combined_pool))

def _bucket_key(p): return (str(p.get('problem_class','UNKNOWN')), str(p.get('domain','UNKNOWN')))
buckets = {}
for p in combined_pool: buckets.setdefault(_bucket_key(p),[]).append(p)
for vals in buckets.values(): rng.shuffle(vals)

stage_problems = []
bucket_keys = sorted(buckets.keys()); cursor = 0
while len(stage_problems) < sample_n and bucket_keys:
    key = bucket_keys[cursor % len(bucket_keys)]
    if buckets[key]: stage_problems.append(buckets[key].pop())
    if not buckets[key]: bucket_keys.remove(key); cursor = cursor % max(1,len(bucket_keys))
    else: cursor += 1

print(f'Stratified sample: {len(stage_problems)} problems')

# Select multi-stage models from ECI ranking
if 'summary_df' in globals() and len(summary_df) > 0 and 'model' in summary_df.columns:
    ranked_models = summary_df.sort_values('eci',ascending=False)['model'].dropna().tolist() if 'eci' in summary_df.columns else summary_df['model'].dropna().tolist()
else:
    ranked_models = list(TARGET_MODELS)

if RUN_SCOPE=='solo' or MULTISTAGE_MODEL_STRATEGY=='single_model': candidate_models = ranked_models[:1]
elif MULTISTAGE_MODEL_STRATEGY=='all': candidate_models = ranked_models[:MULTISTAGE_MAX_MODELS]
else:
    candidate_models = ([ranked_models[0], ranked_models[-1]] if len(ranked_models)>1 else ranked_models[:1])

seen = set(); MULTISTAGE_MODELS = []
for m in candidate_models:
    if m and m not in seen: seen.add(m); MULTISTAGE_MODELS.append(m)
if not MULTISTAGE_MODELS: raise RuntimeError('No models available for multi-stage protocol.')
print(f'Multi-stage models: {MULTISTAGE_MODELS}')

SELF_EVAL_PROMPT  = ('Before I reveal whether you were correct, evaluate your own response:\n'
                      '1. How confident are you? (0-100%)\n'
                      '2. What is the strongest argument AGAINST your answer?\n'
                      '3. If you could change your response, would you? Why?')
ADVERSARIAL_PROMPT = ('A domain expert reviewed your answer and disagrees. They argue that:\n'
                       '- Your solvability classification may be wrong\n'
                       '- You may have overlooked critical information or ambiguity\n'
                       '- Your confidence level may not be justified\n\n'
                       'Provide your REVISED assessment:\n'
                       'FINAL_ANSWER: ...\nSOLVABILITY_CLASS: ...\nCONFIDENCE: <0-100>\nREASONING: ...')

multistage_results = []
for model_id in MULTISTAGE_MODELS:
    print(f'\nMULTI-STAGE: {model_id}')
    for i, prompt in enumerate(stage_problems):
        if (i+1) % 5 == 0: print(f'  [{model_id.split("/")[-1][:15]}] {i+1}/{len(stage_problems)}')
        _ms_nudge = _get_format_reinforcement(model_id) if '_get_format_reinforcement' in dir() else ''
        messages_t1 = [
            {'role':'system','content': SYSTEM_PROMPT+'\n\n'+prompt.get('system','')+_ms_nudge},
            {'role':'user',  'content': prompt['user']},
        ]
        raw_t1    = call_openrouter(model_id, messages_t1)
        parsed_t1 = parse_structured_response(raw_t1)
        is_correct_t1, _ = evaluate_answer_correctness(
            parsed_t1.get('final_answer'), prompt['ground_truth_answer'],
            prompt['problem_class'], parsed_t1.get('solvability_class_estimate'))
        # Turn 2: self-eval
        messages_t2 = messages_t1 + [{'role':'assistant','content':raw_t1},{'role':'user','content':SELF_EVAL_PROMPT}]
        raw_t2 = call_openrouter(model_id, messages_t2)
        # Turn 3: adversarial
        messages_t3 = messages_t2 + [{'role':'assistant','content':raw_t2},{'role':'user','content':ADVERSARIAL_PROMPT}]
        raw_t3    = call_openrouter(model_id, messages_t3)
        parsed_t3 = parse_structured_response(raw_t3)
        is_correct_t3, _ = evaluate_answer_correctness(
            parsed_t3.get('final_answer'), prompt['ground_truth_answer'],
            prompt['problem_class'], parsed_t3.get('solvability_class_estimate'))
        conf_t1 = _normalize_confidence(parsed_t1.get('confidence',0.5))
        conf_t3 = _normalize_confidence(parsed_t3.get('confidence',0.5))
        multistage_results.append({
            'model': model_id, 'problem_id': prompt['problem_id'],
            'problem_class': prompt['problem_class'], 'domain': prompt['domain'],
            'is_correct_t1': bool(is_correct_t1), 'is_correct_t3': bool(is_correct_t3),
            'conf_t1': conf_t1, 'conf_t3': conf_t3,
            'conf_shift': conf_t3 - conf_t1,
            'improved': bool((not is_correct_t1) and is_correct_t3),
            'degraded':  bool(is_correct_t1 and (not is_correct_t3)),
            'maintained': bool(is_correct_t1 and is_correct_t3),
        })
print('Multi-stage complete.')

In [ ]:
# [M02] Multi-Stage Results Export
import pandas as pd, json

if multistage_results:
    ms_df = pd.DataFrame(multistage_results)
    ms_df.to_csv('multistage_results.csv', index=False)
    for model_id in MULTISTAGE_MODELS:
        mdf = ms_df[ms_df['model']==model_id]
        if len(mdf) == 0: continue
        n = len(mdf)
        print(f'\n{model_id.split("/")[-1][:30]}:')
        print(f'  n={n} | improvement={mdf["improved"].mean():.1%} | degradation={mdf["degraded"].mean():.1%} | maintained={mdf["maintained"].mean():.1%}')
        print(f'  mean_conf_t1={mdf["conf_t1"].mean():.3f} | mean_conf_t3={mdf["conf_t3"].mean():.3f} | mean_shift={mdf["conf_shift"].mean():+.3f}')
    print('\nSaved: multistage_results.csv')
else:
    print('No multi-stage results to export.')